In [ ]:
# ============================================================
# NOTEBOOK CELL 2
# Plot only for EGU Poster:
#   - read from cache
#   - generate specific figure (0008-02 vs 0008-02_NOCOUPL)
#   - display inline preview for Jupyter Notebook
# ============================================================

import os
import gc
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from tqdm import tqdm

# ------------------------------------------------------------
# Plot controls
# ------------------------------------------------------------
ONLY_DIAGS = ["O3"]

# plot styling / y-limits
MEAN_LINEWIDTH = 5.0
MEMBER_LINEWIDTH_RATIO = 0.3
MEMBER_ALPHA = 0.3
SIGMA_ALPHA = 0.40

O3_YLIM = (60, 140)
TMIN50_YLIM = (175, 255)
U60N10_YLIM = (-40, 80)

# ------------------------------------------------------------
# 路径配置
# ------------------------------------------------------------
DATA_ROOT = "/mnt/soclim0/public_data/weiji"

# CACHE_ROOT 指向原来存放数据的旧目录
CACHE_ROOT = "/home/weiji/restart_exam/code/20260415egu/plots/hindcast/O3/cache"

# 新的海报输出目录
OUT_ROOT = "/home/weiji/restart_exam/code/20260424EGUPOSTER"
FIG_ROOT = OUT_ROOT
os.makedirs(FIG_ROOT, exist_ok=True)

XLIM_START = 0
XLIM_END = 150

# ------------------------------------------------------------
# Figure Specs 配置
# ------------------------------------------------------------
FIGURE_SPECS = [
    {
        "year": "0008", "prefix": "EGU_Poster_Fig_INT_vs_Clim",
        "experiments": [
            ("0008-02",         "INT-3D",  31, "forestgreen"),
            ("0008-02_NOCOUPL", "Clim-3D", 31, "royalblue"),
        ],
    }
]

# ------------------------------------------------------------
# Simple cache loader
# ------------------------------------------------------------
def open_cached_dataarray_local(path):
    da = xr.open_dataarray(path)
    da.load()
    return da

# ------------------------------------------------------------
# Plot function
# ------------------------------------------------------------
def plot_series_figure(ref_data, clim_data, experiments, year,
                       ylabel, ylim, out_subdir, output_filename_prefix,
                       xlim_start=0, xlim_end=150):
    all_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                  "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    all_month_ticks = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]

    xticks = [tick for tick in all_month_ticks if xlim_start <= tick <= xlim_end]
    xtick_labels = [all_months[i] for i, tick in enumerate(all_month_ticks) if tick in xticks]

    fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)

    # --- 1. 绘制参考线 (Reference) 和 气候态 (Climatology) ---
    nref = ref_data.sizes["time"]
    ref_end = min(xlim_end, nref)
    ax.plot(
        np.arange(xlim_start, ref_end),
        ref_data.isel(time=slice(xlim_start, ref_end)).values,
        color="black", linewidth=MEAN_LINEWIDTH, label="Reference",
        zorder=10
    )

    clim_dim = list(clim_data.dims)[0]
    nclim = clim_data.sizes[clim_dim]
    clim_end = min(xlim_end, nclim)
    ax.plot(
        np.arange(xlim_start, clim_end),
        clim_data.isel({clim_dim: slice(xlim_start, clim_end)}).values,
        color="black", linestyle=":", linewidth=MEAN_LINEWIDTH, label="Climatology",
        zorder=9
    )

    # --- 2. 绘制实验组 ---
    member_lw = MEAN_LINEWIDTH * MEMBER_LINEWIDTH_RATIO

    for exp in experiments:
        data = exp["data"]
        label = exp["label"]  
        offset = exp["offset"]
        color = exp["color"]

        if offset >= xlim_end:
            continue

        total = data.sizes["time"]
        start_idx = max(0, xlim_start - offset)
        end_idx = min(total, xlim_end - offset)
        if end_idx <= start_idx:
            continue

        x = np.arange(offset + start_idx, offset + end_idx)
        sub = data.isel(time=slice(start_idx, end_idx))

        if "member" in sub.dims:
            plot_data = sub.transpose("member", "time")
            mean_line = plot_data.mean("member")
            std_line = plot_data.std("member")
            lower = mean_line - std_line
            upper = mean_line + std_line

            for i in range(plot_data.sizes["member"]):
                ax.plot(
                    x, plot_data.isel(member=i).values,
                    color=color, linewidth=member_lw, alpha=MEMBER_ALPHA,
                    zorder=2
                )

            ax.fill_between(x, lower.values, upper.values, color=color, alpha=SIGMA_ALPHA, zorder=1)            
            ax.plot(x, lower.values, color=color, linewidth=1, alpha=1, zorder=2)
            ax.plot(x, upper.values, color=color, linewidth=1, alpha=1, zorder=2)
            ax.plot(x, mean_line.values, color=color, linewidth=MEAN_LINEWIDTH, label=label, zorder=3)
        else:
            ax.plot(x, sub.values, color=color, linewidth=MEAN_LINEWIDTH, label=label, zorder=3)

    # --- 3. 细节微调 ---
    ax.set_xticks(xticks)
    ax.set_xticklabels(xtick_labels, fontsize=16)
    ax.set_xlim(xlim_start, xlim_end)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.set_ylim(*ylim)
    ax.tick_params(axis="y", labelsize=16)

    ax.text(
        0.02, 0.95, f"Year: {year}",
        transform=ax.transAxes, fontsize=18, verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5)
    )

    ax.legend(fontsize=15) 

    fig_dir = os.path.join(FIG_ROOT, out_subdir)
    os.makedirs(fig_dir, exist_ok=True)

    base_filepath = os.path.join(fig_dir, f"{output_filename_prefix}_{year}")

    # --- 4. 保存图片 ---
    fig.savefig(f"{base_filepath}.pdf", bbox_inches="tight")
    fig.savefig(f"{base_filepath}.tif", dpi=500, format="tiff", bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})

    return fig, ax

# ------------------------------------------------------------
# Load cache + plot
# ------------------------------------------------------------
def plot_suite_from_cache(diag_name, ylabel, ylim, out_subdir):
    print(f"\n================ PLOT {diag_name} FROM CACHE =================\n")

    years_needed = sorted(set(spec["year"] for spec in FIGURE_SPECS))
    all_exp_names = sorted(set(exp_name for spec in FIGURE_SPECS for exp_name, _, _, _ in spec["experiments"]))

    clim_path = os.path.join(CACHE_ROOT, diag_name, f"{diag_name}_B2000WCN_climatology.nc")
    if not os.path.exists(clim_path):
        raise FileNotFoundError(f"Missing climatology cache: {clim_path}")

    clim = open_cached_dataarray_local(clim_path)

    ref_cache = {}
    for year in years_needed:
        ref_path = os.path.join(CACHE_ROOT, diag_name, f"{diag_name}_BWCN_{year}.nc")
        if not os.path.exists(ref_path):
            raise FileNotFoundError(f"Missing BWCN cache: {ref_path}")
        ref_cache[year] = open_cached_dataarray_local(ref_path)

    exp_cache = {}
    for exp_name in all_exp_names:
        exp_path = os.path.join(CACHE_ROOT, diag_name, f"{diag_name}_Hindcast_{exp_name}.nc")
        if not os.path.exists(exp_path):
            raise FileNotFoundError(f"Missing Hindcast cache: {exp_path}")
        exp_cache[exp_name] = open_cached_dataarray_local(exp_path)

    for spec in tqdm(FIGURE_SPECS, desc=f"Plotting {diag_name}"):
        plot_prefix = spec["prefix"]
        if diag_name != "O3":
            plot_prefix = plot_prefix.replace("O3", f"{diag_name}")

        experiments = []
        for exp_name, label, offset, color in spec["experiments"]:
            experiments.append({
                "data": exp_cache[exp_name],
                "label": label,
                "offset": offset,
                "color": color,
            })

        fig, ax = plot_series_figure(
            ref_data=ref_cache[spec["year"]],
            clim_data=clim,
            experiments=experiments,
            year=spec["year"],
            ylabel=ylabel,
            ylim=ylim,
            out_subdir=out_subdir,
            output_filename_prefix=plot_prefix,
            xlim_start=XLIM_START,
            xlim_end=XLIM_END,
        )
        
        # --- 核心更新：在此处展示图片，让你在 ipynb 中直接预览 ---
        plt.show()
        
        # 预览完后关闭画布释放内存
        plt.close(fig)
        gc.collect()

# ------------------------------------------------------------
# Run plotting
# ------------------------------------------------------------
if "O3" in ONLY_DIAGS:
    plot_suite_from_cache(
        diag_name="O3",
        ylabel="Partial ozone column, 30–70 hPa (DU)",
        ylim=O3_YLIM,
        out_subdir="O3",
    )

if "Tmin50" in ONLY_DIAGS:
    plot_suite_from_cache(
        diag_name="Tmin50",
        ylabel="Polar minimum T at 50 hPa (K)",
        ylim=TMIN50_YLIM,
        out_subdir="Tmin50",
    )

if "U60N10" in ONLY_DIAGS:
    plot_suite_from_cache(
        diag_name="U60N10",
        ylabel="Zonal-mean U at 60°N, 10 hPa (m s$^{-1}$)",
        ylim=U60N10_YLIM,
        out_subdir="U60N10",
    )

print("\nPoster figure successfully generated to the new directory and displayed above.")

In [ ]:
# ============================================================
# NOTEBOOK CELL: Figure 3cd
# Pooled B2000WCN/BWCN vs Reanalysis for EGU Poster
# 修正版：
#   1. 修复 MERRA2 EP flux pressure level 选择错误
#   2. 自动判断 plev 单位是 hPa 还是 Pa
#   3. WACCM 使用 -Fz，MERRA2/Reanalysis 使用 +Fz
#   4. 输出 dpi=500 的 tif + pdf + debug png，并 plt.show()
# ============================================================

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

# ================= 1. 配置区 =================
OUT_ROOT = "/home/weiji/restart_exam/code/20260424EGUPOSTER"
os.makedirs(OUT_ROOT, exist_ok=True)

OUT_FIG_BASE = os.path.join(OUT_ROOT, "Fig3_cd_2x2_WACCM_vs_Reanalysis_Poster")
OUT_FIG_TIF  = OUT_FIG_BASE + ".tif"
OUT_FIG_PDF  = OUT_FIG_BASE + ".pdf"
OUT_FIG_PNG  = OUT_FIG_BASE + "_debug.png"

# --- BWCN 路径 ---
BWCN_DIR = "/mnt/soclim0/public_data/weiji/BWCN"
BWCN_NAM = os.path.join(BWCN_DIR, "NAM", "BWCN_Vertical_NAM.nc")
BWCN_AO  = os.path.join(BWCN_DIR, "NAM", "BWCN_Daily_AO_Index.csv")
BWCN_EP  = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
BWCN_O3  = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data/BWCN/O3_pc_BWCN_30_70hPa.nc"

# --- B2000WCN 路径 ---
B2000_DIR = "/mnt/soclim0/public_data/weiji/B2000WCN001002"
B2000_NAM = os.path.join(B2000_DIR, "NAM", "B2000WCN001002_Vertical_NAM.nc")
B2000_AO  = os.path.join(B2000_DIR, "NAM", "B2000WCN001002_Daily_AO_Index.csv")
B2000_EP  = os.path.join(B2000_DIR, "EPflux_daily", "EPFLUX_206yr_Daily_Series_Full.nc")
B2000_O3  = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data/B2000WCN/O3_pc_B2000WCN_30_70hPa.nc"

# --- Reanalysis 路径 ---
# 注意：这里右列实际是 ERA5 NAM/AO + MERRA2 Fz/O3
MERRA2_DIR = "/mnt/soclim0/public_data/weiji/MERRA2_Processed"

MERRA2_AO  = "/home/weiji/restart_exam/code/20260324_AO_compare_A_vs_Bstyle_STRICT/AO_codeA.nc"
MERRA2_NAM = "/home/weiji/restart_exam/code/20260315NAM/resul/New_NAM_Algorithm/ERA5_daily_vertical_NAM_monthlyEOF_DOYprojected.nc"

MERRA2_EP  = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"
MERRA2_O3  = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data/MERRA2/O3_pc_MERRA2_30_70hPa.nc"

LAT_FZ = (40.0, 80.0)


# ================= 2. 核心辅助函数 =================
def sel_lat_safe(da, lat_range):
    lat0, lat1 = lat_range
    if "lat" not in da.coords:
        return da

    if da["lat"].values[0] > da["lat"].values[-1]:
        return da.sel(lat=slice(lat1, lat0))
    return da.sel(lat=slice(lat0, lat1))


def get_coslat_weight(da, lat_range):
    da_sub = sel_lat_safe(da, lat_range)
    weights = np.cos(np.deg2rad(da_sub["lat"]))
    return da_sub.weighted(weights).mean("lat", skipna=True)


def select_pressure_level_safe(da, coord_name, target_hpa, exp_name="", var_name=""):
    """
    安全选择 pressure level。

    规则：
      - 如果坐标最大值 > 2000，则认为单位是 Pa，例如 10000 = 100 hPa
      - 否则认为单位是 hPa，例如 100 = 100 hPa

    target_hpa 永远用 hPa 输入。
    """
    vals = da[coord_name].values.astype(float)
    vmax = np.nanmax(vals)

    if vmax > 2000:
        target = target_hpa * 100.0
        unit_guess = "Pa"
    else:
        target = target_hpa
        unit_guess = "hPa"

    selected = float(da[coord_name].sel({coord_name: target}, method="nearest").values)

    print(
        f"    [{exp_name}] {var_name}: coord={coord_name}, "
        f"unit_guess={unit_guess}, target={target_hpa:g} hPa, "
        f"selected={selected:g}"
    )

    return da.sel({coord_name: target}, method="nearest")


def get_djf_mean_robust(da):
    """
    DJF mean with DJF year assigned to Jan/Feb year.
    Example:
      Dec 1980 + Jan 1981 + Feb 1981 -> year 1981
    """
    years = da["time"].dt.year
    months = da["time"].dt.month

    djf_years = xr.where(months == 12, years + 1, years)
    da = da.assign_coords(djf_year=djf_years)

    da_djf = da.sel(time=da["time"].dt.month.isin([12, 1, 2]))
    return da_djf.groupby("djf_year").mean("time").rename({"djf_year": "year"})


def get_jfm_mean_native(da):
    jfm = da.sel(time=da["time"].dt.month.isin([1, 2, 3]))
    return jfm.groupby("time.year").mean("time").rename({"year": "year"})


def get_ma_min_native(da):
    ma = da.sel(time=da["time"].dt.month.isin([3, 4]))
    return ma.groupby("time.year").min("time").rename({"year": "year"})


def standardize_column(df, cols, prefix_note=""):
    for col in cols:
        mu = df[col].mean()
        sd = df[col].std(ddof=0)

        if not np.isfinite(sd) or sd == 0:
            raise ValueError(f"{prefix_note} Cannot standardize {col}: std={sd}")

        df[col.replace("_raw", "")] = (df[col] - mu) / sd
    return df


# ================= 3. 数据提取逻辑 =================
def prep_one_experiment(exp_name, nam_nc, ao_file, ep_nc, o3_nc):
    print(f"\n🔄 正在处理并对齐实验数据: {exp_name}...")

    # ------------------------------------------------------------
    # 1. NAM: JFM 50 hPa
    # ------------------------------------------------------------
    ds_nam = xr.open_dataset(nam_nc, use_cftime=True)
    var_name = "NAM" if "NAM" in ds_nam.data_vars else "NAM_Vertical"

    if "lev" in ds_nam[var_name].coords:
        lev_coord = "lev"
    elif "level" in ds_nam[var_name].coords:
        lev_coord = "level"
    elif "plev" in ds_nam[var_name].coords:
        lev_coord = "plev"
    else:
        raise ValueError(f"[{exp_name}] Cannot find NAM vertical coordinate.")

    nam_raw = select_pressure_level_safe(
        ds_nam[var_name],
        coord_name=lev_coord,
        target_hpa=50.0,
        exp_name=exp_name,
        var_name=var_name
    )

    da_nam = get_jfm_mean_native(nam_raw)

    # ------------------------------------------------------------
    # 2. EP flux: DJF 100 hPa 40–80N
    # ------------------------------------------------------------
    ds_ep = xr.open_dataset(ep_nc, use_cftime=True)
    var_fz = "Fz" if "Fz" in ds_ep.data_vars else "EP2_cart"

    fz_lev_coord = next(
        c for c in ["plev", "lev", "level"]
        if c in ds_ep[var_fz].coords or c in ds_ep[var_fz].dims
    )

    fz_raw = select_pressure_level_safe(
        ds_ep[var_fz],
        coord_name=fz_lev_coord,
        target_hpa=100.0,
        exp_name=exp_name,
        var_name=var_fz
    )

    # 如果还有 lat 维度，则做 40–80N cos-weighted mean
    # MERRA2 文件已经是 40–80N mean，所以不会重复平均
    if "lat" in fz_raw.coords:
        fz_raw = get_coslat_weight(fz_raw, LAT_FZ)

    # ------------------------------------------------------------
    # Fz 符号约定
    # ------------------------------------------------------------
    # WACCM 继续沿用你原来代码的 -Fz；
    # MERRA2/Reanalysis 根据 debug 结果使用 +Fz：
    #   100 hPa +Fz vs ERA5 NAM50: r ≈ +0.707
    # 如果你之后想统一物理符号，需要单独检查 WACCM 和 MERRA2 的 EP flux 定义。

    fz_raw = -1.0 * fz_raw


    da_fz = get_djf_mean_robust(fz_raw)

    # ------------------------------------------------------------
    # B2000WCN EPFLUX 年份错位补丁
    # ------------------------------------------------------------
    if exp_name == "B2000WCN":
        fz_years = da_fz.year.values.astype(int)
        corrected_years = np.where(fz_years >= 105, fz_years + 1, fz_years)
        da_fz = da_fz.assign_coords(year=corrected_years)
        print("    [B2000WCN] Applied EPFLUX year correction: years >=105 shifted to +1.")

    # ------------------------------------------------------------
    # 过滤 NAM 幽灵 104 年 / Fz 104 年
    # ------------------------------------------------------------
    if 104 in da_nam.year.values:
        da_nam = da_nam.sel(year=(da_nam.year != 104))
        print("    Removed ghost year 104 from NAM.")

    if 104 in da_fz.year.values:
        da_fz = da_fz.sel(year=(da_fz.year != 104))
        print("    Removed year 104 from Fz.")

    # ------------------------------------------------------------
    # 3. AO: JFM mean
    # ------------------------------------------------------------
    if ao_file.endswith(".nc"):
        ds_ao_nc = xr.open_dataset(ao_file, use_cftime=True)
        ao_var = "AO_Index" if "AO_Index" in ds_ao_nc.data_vars else list(ds_ao_nc.data_vars)[0]
        da_ao = get_jfm_mean_native(ds_ao_nc[ao_var])
        ds_ao_nc.close()

    else:
        df_ao = pd.read_csv(ao_file)
        df_ao["Date"] = df_ao["Date"].astype(str)
        df_ao["year"] = df_ao["Date"].str.slice(0, 4).astype(int)
        df_ao["month"] = df_ao["Date"].str.slice(5, 7).astype(int)

        df_ao_jfm = df_ao[df_ao["month"].isin([1, 2, 3])]
        ao_jfm_yr = df_ao_jfm.groupby("year")["AO_Index"].mean()

        da_ao = xr.DataArray(
            ao_jfm_yr.values,
            coords={"year": ao_jfm_yr.index.values.astype(int)},
            dims=["year"]
        )

    # ------------------------------------------------------------
    # 4. O3: Mar–Apr minimum after 5-day rolling mean
    # ------------------------------------------------------------
    da_o3 = xr.open_dataarray(o3_nc, use_cftime=True)
    da_o3 = da_o3.rolling(time=5, center=True, min_periods=5).mean()
    da_o3 = get_ma_min_native(da_o3)

    # ------------------------------------------------------------
    # 5. 合并共同年份
    # ------------------------------------------------------------
    common_years = np.intersect1d(da_nam.year.values.astype(int), da_fz.year.values.astype(int))
    common_years = np.intersect1d(common_years, da_ao.year.values.astype(int))
    common_years = np.intersect1d(common_years, da_o3.year.values.astype(int))
    common_years = common_years[common_years != 104]

    print(
        f"    [{exp_name}] common years: "
        f"{common_years.min() if len(common_years) else 'NA'}–"
        f"{common_years.max() if len(common_years) else 'NA'}, "
        f"n={len(common_years)}"
    )

    df = pd.DataFrame({
        "exp": exp_name,
        "year": common_years.astype(int),
        "nam_raw": da_nam.sel(year=common_years).values,
        "fz_raw":  da_fz.sel(year=common_years).values,
        "ao_raw":  da_ao.sel(year=common_years).values,
        "o3_raw":  da_o3.sel(year=common_years).values,
    })

    ds_nam.close()
    ds_ep.close()

    return df


# ================= 4. WACCM 数据准备 =================
df_bwcn = prep_one_experiment(
    "BWCN",
    BWCN_NAM,
    BWCN_AO,
    BWCN_EP,
    BWCN_O3
)

df_b2000 = prep_one_experiment(
    "B2000WCN",
    B2000_NAM,
    B2000_AO,
    B2000_EP,
    B2000_O3
)

df_waccm = pd.concat([df_bwcn, df_b2000], ignore_index=True).dropna()

df_waccm = standardize_column(
    df_waccm,
    ["nam_raw", "fz_raw", "ao_raw"],
    prefix_note="WACCM"
)

df_top5_waccm = df_waccm.sort_values("o3_raw", ascending=True).head(5).copy()
df_top5_waccm["rank"] = range(1, 6)

print("\n🏆 WACCM 全局 Top 5 最低臭氧年份:")
print(df_top5_waccm[["rank", "exp", "year", "o3_raw"]].to_string(index=False))


# ================= 5. Reanalysis 数据准备 =================
df_merra2 = prep_one_experiment(
    "MERRA2",
    MERRA2_NAM,
    MERRA2_AO,
    MERRA2_EP,
    MERRA2_O3
)

df_merra2 = df_merra2.dropna()

df_merra2 = standardize_column(
    df_merra2,
    ["nam_raw", "fz_raw", "ao_raw"],
    prefix_note="MERRA2"
)

df_top5_merra2 = df_merra2.sort_values("o3_raw", ascending=True).head(5).copy()
df_top5_merra2["rank"] = range(1, 6)

print("\n🏆 MERRA2 Top 5 最低臭氧年份:")
print(df_top5_merra2[["rank", "exp", "year", "o3_raw"]].to_string(index=False))


# ================= 6. 打印相关性检查 =================
def print_corr_check(label, df):
    r_fz, p_fz = pearsonr(df["nam"], df["fz"])
    r_ao, p_ao = pearsonr(df["nam"], df["ao"])

    print(f"\n📌 Correlation check: {label}")
    print(f"    NAM50 vs Fz : r = {r_fz:.3f}, p = {p_fz:.3g}, n = {len(df)}")
    print(f"    NAM50 vs AO : r = {r_ao:.3f}, p = {p_ao:.3g}, n = {len(df)}")


print_corr_check("WACCM pooled", df_waccm)
print_corr_check("Reanalysis / MERRA2+ERA5", df_merra2)


# ================= 7. 绘图 =================
print(f"\n🎨 开始绘制 WACCM vs Reanalysis 对比图...")

plt.rcParams["figure.facecolor"] = "w"
plt.rcParams["savefig.facecolor"] = "w"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

fig, axes = plt.subplots(
    2, 2,
    figsize=(12.5, 11),
    sharex="col",
    sharey="row"
)

plt.subplots_adjust(
    hspace=0.08,
    wspace=0.10,
    bottom=0.20,
    top=0.92
)

severity_colors = ["darkred", "red", "gold", "royalblue", "navy"]
limit = (-3, 3)


def apply_axes_style(ax, ylabel, show_xticklabels=True, show_yticklabels=True, xlabel="JFM, 50 hPa NAM (σ)"):
    if show_xticklabels:
        ax.set_xlabel(xlabel, fontsize=13)

    if show_yticklabels:
        ax.set_ylabel(ylabel, fontsize=13)

    ax.set_xlim(limit[0], limit[1])
    ax.set_ylim(limit[0], limit[1])

    ax.set_xticks(np.arange(-3, 4, 1))
    ax.set_yticks(np.arange(-3, 4, 1))

    if not show_xticklabels:
        ax.tick_params(axis="x", labelbottom=False)

    if not show_yticklabels:
        ax.tick_params(axis="y", labelleft=False)

    ax.grid(True, linestyle=":", color="gray", alpha=0.6)
    ax.axhline(0, color="gray", linewidth=0.8, zorder=1)
    ax.axvline(0, color="gray", linewidth=0.8, zorder=1)


def plot_scatter_and_fit(
    ax,
    df_data,
    df_top,
    y_col,
    ylabel,
    show_xticklabels,
    show_yticklabels,
    is_waccm=True,
    xlabel="JFM, 50 hPa NAM (σ)"
):
    # -----------------------------
    # 基础点
    # -----------------------------
    if is_waccm:
        m_b2000 = df_data["exp"] == "B2000WCN"
        m_bwcn  = df_data["exp"] == "BWCN"

        ax.scatter(
            df_data.loc[m_b2000, "nam"],
            df_data.loc[m_b2000, y_col],
            color="0.2",
            s=40,
            alpha=0.35,
            edgecolors="none",
            zorder=3
        )

        ax.scatter(
            df_data.loc[m_bwcn, "nam"],
            df_data.loc[m_bwcn, y_col],
            s=40,
            alpha=0.60,
            facecolors="none",
            edgecolors="0.2",
            linewidths=0.8,
            zorder=3
        )

    else:
        ax.scatter(
            df_data["nam"],
            df_data[y_col],
            color="0.2",
            s=40,
            alpha=0.50,
            edgecolors="none",
            zorder=3
        )

    # -----------------------------
    # Top 5 低臭氧年份
    # -----------------------------
    for i, (_, r) in enumerate(df_top.iterrows()):
        c = severity_colors[i]

        if is_waccm and r["exp"] == "BWCN":
            ax.scatter(
                r["nam"],
                r[y_col],
                s=130,
                facecolors="none",
                edgecolors=c,
                linewidths=2.0,
                zorder=10
            )
        else:
            ax.scatter(
                r["nam"],
                r[y_col],
                s=130,
                color=c,
                edgecolors="k",
                linewidths=0.9,
                zorder=10
            )

    # -----------------------------
    # 相关性与回归线
    # -----------------------------
    clean = df_data[["nam", y_col]].replace([np.inf, -np.inf], np.nan).dropna()

    r_val, p_val = pearsonr(clean["nam"], clean[y_col])
    star = "*" if p_val < 0.05 else ""

    m, b = np.polyfit(clean["nam"], clean[y_col], 1)
    x_line = np.linspace(limit[0] - 0.2, limit[1] + 0.2, 100)

    ax.plot(
        x_line,
        m * x_line + b,
        color="gray",
        linestyle="-",
        lw=1.2,
        alpha=0.6,
        zorder=2
    )

    ax.text(
        0.05,
        0.95,
        f"r = {r_val:.3f}{star}",
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="left",
        zorder=10,
        bbox=dict(boxstyle="round", fc="w", alpha=0.6, edgecolor="none")
    )

    apply_axes_style(
        ax,
        ylabel=ylabel,
        show_xticklabels=show_xticklabels,
        show_yticklabels=show_yticklabels,
        xlabel=xlabel
    )


# -----------------------------
# 2x2 panels
# -----------------------------

# [0, 0] WACCM - Fz
plot_scatter_and_fit(
    axes[0, 0],
    df_waccm,
    df_top5_waccm,
    "fz",
    "DJF, 100 hPa 40–80°N EP-flux metric (σ)",
    show_xticklabels=False,
    show_yticklabels=True,
    is_waccm=True,
    xlabel="JFM, 50 hPa NAM (Pooled σ)"
)

axes[0, 0].set_title(
    "WACCM4 INT 3D (Pooled)",
    fontsize=16,
    fontweight="bold",
    pad=15
)

# [0, 1] Reanalysis - Fz
plot_scatter_and_fit(
    axes[0, 1],
    df_merra2,
    df_top5_merra2,
    "fz",
    "",
    show_xticklabels=False,
    show_yticklabels=False,
    is_waccm=False,
    xlabel="JFM, 50 hPa NAM (σ)"
)

axes[0, 1].set_title(
    "Reanalysis",
    fontsize=16,
    fontweight="bold",
    pad=15
)

# [1, 0] WACCM - AO
plot_scatter_and_fit(
    axes[1, 0],
    df_waccm,
    df_top5_waccm,
    "ao",
    "JFM, AO Index (σ)",
    show_xticklabels=True,
    show_yticklabels=True,
    is_waccm=True,
    xlabel="JFM, 50 hPa NAM (Pooled σ)"
)

# [1, 1] Reanalysis - AO
plot_scatter_and_fit(
    axes[1, 1],
    df_merra2,
    df_top5_merra2,
    "ao",
    "",
    show_xticklabels=True,
    show_yticklabels=False,
    is_waccm=False,
    xlabel="JFM, 50 hPa NAM (σ)"
)


# ================= 8. 底部图例 =================

legend_elements_waccm = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        label="B2000WCN (All)",
        markerfacecolor="0.2",
        markersize=8,
        alpha=0.5,
        markeredgecolor="none"
    ),
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        label="BWCN (All)",
        markerfacecolor="none",
        markersize=8,
        alpha=0.8,
        markeredgecolor="0.2",
        markeredgewidth=1
    ),
]

for i, (_, r) in enumerate(df_top5_waccm.iterrows()):
    c = severity_colors[i]
    label_text = f"Top {i+1}: {r['exp']}-{int(r['year']):04d}"

    if r["exp"] == "B2000WCN":
        leg = Line2D(
            [0], [0],
            marker="o",
            color="none",
            label=label_text,
            markerfacecolor=c,
            markersize=10,
            markeredgecolor="k",
            markeredgewidth=0.9
        )
    else:
        leg = Line2D(
            [0], [0],
            marker="o",
            color="none",
            label=label_text,
            markerfacecolor="none",
            markersize=10,
            markeredgecolor=c,
            markeredgewidth=2.0
        )

    legend_elements_waccm.append(leg)


legend_elements_merra2 = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        label="Reanalysis (All)",
        markerfacecolor="0.2",
        markersize=8,
        alpha=0.5,
        markeredgecolor="none"
    )
]

for i, (_, r) in enumerate(df_top5_merra2.iterrows()):
    c = severity_colors[i]
    label_text = f"Top {i+1}: Year-{int(r['year']):04d}"

    leg = Line2D(
        [0], [0],
        marker="o",
        color="none",
        label=label_text,
        markerfacecolor=c,
        markersize=10,
        markeredgecolor="k",
        markeredgewidth=0.9
    )

    legend_elements_merra2.append(leg)


fig.legend(
    handles=legend_elements_waccm,
    title="WACCM Global Lowest O$_3$ Years",
    fontsize=10,
    title_fontsize=11,
    loc="lower center",
    bbox_to_anchor=(0.30, 0.01),
    ncol=2,
    frameon=True,
    facecolor="w",
    edgecolor="lightgray"
)

fig.legend(
    handles=legend_elements_merra2,
    title="Reanalysis Lowest O$_3$ Years",
    fontsize=10,
    title_fontsize=11,
    loc="lower center",
    bbox_to_anchor=(0.70, 0.01),
    ncol=2,
    frameon=True,
    facecolor="w",
    edgecolor="lightgray"
)


# ================= 9. 保存输出 =================

# debug PNG
fig.savefig(
    OUT_FIG_PNG,
    dpi=500,
    bbox_inches="tight",
    facecolor="white"
)

# poster TIFF
fig.savefig(
    OUT_FIG_TIF,
    dpi=500,
    bbox_inches="tight",
    facecolor="white",
    format="tiff"
)

# vector PDF
fig.savefig(
    OUT_FIG_PDF,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("\n✅ WACCM 与 Reanalysis 并排对比绘图完成！")
print(f"   PNG debug : {OUT_FIG_PNG}")
print(f"   TIFF 500dpi: {OUT_FIG_TIF}")
print(f"   PDF vector: {OUT_FIG_PDF}")

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ============================================================
# 1. 路径设置
# ============================================================
HINDCAST_BASE = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01"
CSV_IN = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01/O3_eval/Hindcast_0008-01_O3_RMSE_Ranking.csv"

# 输出路径改到 EGUPOSTER
PLOT_OUT = "/home/weiji/restart_exam/code/20260424EGUPOSTER"
os.makedirs(PLOT_OUT, exist_ok=True)

OUT_BASE = os.path.join(PLOT_OUT, "Scatter_RMSE_vs_Fz_Jan20-Feb10")
OUT_PNG  = OUT_BASE + "_debug.png"
OUT_TIF  = OUT_BASE + ".tif"
OUT_PDF  = OUT_BASE + ".pdf"

# ============================================================
# 2. 提取 100 hPa Fz 平均值 (Jan 20 - Feb 10)
# ============================================================
def get_fz_mean_jan20_feb10(member_id):
    """
    提取单成员 1月20日到2月10日的 100 hPa EP Flux Fz 平均值。

    注意：
    - 若 plev 单位为 Pa，则选择 10000 Pa；
    - 若 plev 单位为 hPa，则选择 100 hPa；
    - 最后乘以 -1，使 upward wave activity metric 为正向表达。
    """
    mid_str = str(member_id).zfill(3)
    folder = os.path.join(HINDCAST_BASE, "EPflux_daily")
    files = glob.glob(os.path.join(folder, f"*.{mid_str}.*.nc"))

    if not files:
        return np.nan

    ds = xr.open_dataset(files[0])

    try:
        nc_var = "Fz" if "Fz" in ds.data_vars else ("EPFLUX" if "EPFLUX" in ds.data_vars else list(ds.data_vars)[0])

        mask_jan = (ds.time.dt.month == 1) & (ds.time.dt.day >= 20)
        mask_feb = (ds.time.dt.month == 2) & (ds.time.dt.day <= 10)
        mask_time = mask_jan | mask_feb

        ds_sub = ds.isel(time=mask_time)
        da = ds_sub[nc_var]

        if "lev" in da.dims:
            da = da.rename({"lev": "plev"})

        if "plev" in da.dims:
            if float(da["plev"].max().values) > 2000:
                val_100hpa = da.sel(plev=10000.0, method="nearest")
            else:
                val_100hpa = da.sel(plev=100.0, method="nearest")
        else:
            val_100hpa = da

        # 保持你原来的物理约定：使用 -Fz 表示 upward wave activity metric
        fz_mean = -val_100hpa.mean(dim="time", skipna=True).values

        return float(fz_mean)

    finally:
        ds.close()


# ============================================================
# 3. 数据配对处理
# ============================================================
print("[INFO] 正在读取 RMSE 排名并提取 EP Flux 数据...")

df = pd.read_csv(CSV_IN, dtype={"Member": str})

x_fz = []
y_rmse = []
valid_members = []

for idx, row in df.iterrows():
    mid = row["Member"]
    rmse = row["RMSE_DU"]

    fz_mean = get_fz_mean_jan20_feb10(mid)

    if np.isfinite(fz_mean):
        x_fz.append(fz_mean)
        y_rmse.append(rmse)
        valid_members.append(mid)
    else:
        print(f"  [警告] 成员 {mid} 获取 Fz 数据失败。")

x_fz = np.array(x_fz, dtype=float)
y_rmse = np.array(y_rmse, dtype=float)

if len(x_fz) < 2:
    raise ValueError("有效成员少于 2 个，无法计算回归。")


# ============================================================
# 4. 绘制散点图与回归线
# ============================================================
plt.rcParams["figure.facecolor"] = "w"
plt.rcParams["savefig.facecolor"] = "w"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

fig, ax = plt.subplots(figsize=(7.5, 5.5), constrained_layout=True)

ax.scatter(
    x_fz,
    y_rmse,
    s=60,
    c="dodgerblue",
    alpha=0.8,
    edgecolor="k",
    linewidth=0.5,
    zorder=3
)

# 回归线与统计量
slope, intercept, r_value, p_value, std_err = linregress(x_fz, y_rmse)

x_margin = (x_fz.max() - x_fz.min()) * 0.10
y_margin = (y_rmse.max() - y_rmse.min()) * 0.10

if x_margin == 0:
    x_margin = abs(x_fz.mean()) * 0.05 if x_fz.mean() != 0 else 1.0

if y_margin == 0:
    y_margin = abs(y_rmse.mean()) * 0.05 if y_rmse.mean() != 0 else 1.0

x_fit = np.linspace(x_fz.min() - x_margin, x_fz.max() + x_margin, 100)
y_fit = slope * x_fit + intercept

# 删除 legend 里的 Linear Trend 表述：不设置 label，也不调用 legend
ax.plot(
    x_fit,
    y_fit,
    color="crimson",
    lw=2,
    ls="--",
    zorder=2
)

# 只保留 R 和 P，不显示 R2
text_str = f"R = {r_value:.3f}\nP = {p_value:.3e}"
props = dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="gray")

# 放在右上角
ax.text(
    0.95,
    0.95,
    text_str,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="top",
    horizontalalignment="right",
    bbox=props
)

# 标签修改
ax.set_xlabel(
    "Mean 100 hPa EP Flux $-F_z$ "
    "(40–80°N average, Jan 20–Feb 10)",
    fontsize=11
)

ax.set_ylabel(
    "O$_3$ Partial Column RMSE (DU; evaluation period from Jan 20 to Feb 10)",
    fontsize=11
)

ax.set_title(
    "Ozone RMSE vs. Winter Wave Activity",
    fontsize=13
)

ax.set_xlim(x_fz.min() - x_margin, x_fz.max() + x_margin)
ax.set_ylim(y_rmse.min() - y_margin, y_rmse.max() + y_margin)

ax.grid(True, linestyle="--", alpha=0.6, zorder=1)

# ============================================================
# 5. 保存输出：PNG debug + TIFF 500 dpi + PDF
# ============================================================
fig.savefig(OUT_PNG, dpi=500, bbox_inches="tight", facecolor="white")
fig.savefig(OUT_TIF, dpi=500, bbox_inches="tight", facecolor="white", format="tiff")
fig.savefig(OUT_PDF, bbox_inches="tight", facecolor="white")

plt.show()

print("\n[SUCCESS] 散点图绘制完毕！")
print(f"PNG debug : {OUT_PNG}")
print(f"TIFF 500dpi: {OUT_TIF}")
print(f"PDF vector: {OUT_PDF}")

In [ ]:
import os
import glob
import gc
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# EGU POSTER ONLY
# O3 partial column figure:
#   Hindcast 0008-01 ensemble
#   + target-year reference
#   + B2000WCN climatology from the same cache as previous poster figure
#   + best/worst members based on O3 RMSE ranking
#   + save PDF / TIF / PNG
#   + show inline preview in Jupyter Notebook
# ============================================================

# ============================================================
# 1. 路径与参数设置
# ============================================================
HINDCAST_BASE = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01"
CSV_IN = "/mnt/soclim0/public_data/weiji/Hindcast/0008-01/O3_eval/Hindcast_0008-01_O3_RMSE_Ranking.csv"

# Target-year reference file
REF_O3_FILE = "/mnt/soclim0/public_data/weiji/BWCN/O3/BWCN.cam.h3.0008.O3.nc"

# 与上一张 poster 图完全一致的 climatology cache
CACHE_ROOT = "/home/weiji/restart_exam/code/20260415egu/plots/hindcast/O3/cache"
CLIM_O3_FILE = os.path.join(CACHE_ROOT, "O3", "O3_B2000WCN_climatology.nc")

# 新的 EGU poster 输出目录
OUT_ROOT = "/home/weiji/restart_exam/code/20260424EGUPOSTER"
PLOT_OUT = os.path.join(OUT_ROOT, "O3_Hindcast_0008-01")
os.makedirs(PLOT_OUT, exist_ok=True)

# 输出文件名前缀
OUT_PREFIX = "EGU_Poster_O3_Column_Evolution_0008-01"

COMPARE_N = 5

# ============================================================
# 2. 绘图控制
# ============================================================
O3_YLIM = (60, 140)

# 原代码字体大约 1.5 倍
TITLE_FONTSIZE = 20      # 原 13 左右
LABEL_FONTSIZE = 17      # 原 11
TICK_FONTSIZE = 15       # 原默认/约10
LEGEND_FONTSIZE = 13.5   # 原 9

# 线宽也适当放大，便于 poster 使用
MEMBER_LW = 0.9
MEAN_LW = 3.75
REF_LW = 4.5
CLIM_LW = 4.0
SIGMA_ALPHA = 0.20
MEMBER_ALPHA = 0.35

# ============================================================
# 3. 时间筛选函数
# ============================================================
def _get_time_mask(ds):
    """
    Select Jan 1 to May 30.
    This is consistent with the original hindcast plotting code.
    """
    mask = (ds.time.dt.month >= 1) & (ds.time.dt.month <= 5)
    mask = mask & ~((ds.time.dt.month == 5) & (ds.time.dt.day == 31))
    return mask

# ============================================================
# 4. O3 partial column calculation
# ============================================================
def calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa):
    """
    Calculate partial ozone column from hybrid-sigma levels.

    Input:
        ds_sub: dataset containing O3, PS, P0, hyai, hybi
        p_top_hpa, p_bot_hpa: pressure range in hPa

    Output:
        O3 partial column in DU, with dimensions time/lat/lon
    """
    g = 9.80665
    M_air = 28.964 / 1000.0
    Na = 6.02214e23
    factor = Na / (g * M_air * 2.687e20)

    P0 = ds_sub["P0"]
    PS = ds_sub["PS"]

    P_interface = ds_sub["hyai"] * P0 + ds_sub["hybi"] * PS

    p_i = (
        P_interface
        .isel(ilev=slice(0, -1))
        .rename({"ilev": "lev"})
        .assign_coords(lev=ds_sub["lev"])
    )

    p_ip1 = (
        P_interface
        .isel(ilev=slice(1, None))
        .rename({"ilev": "lev"})
        .assign_coords(lev=ds_sub["lev"])
    )

    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)

    pT = p_top_hpa * 100.0
    pB = p_bot_hpa * 100.0

    upper = xr.where(p_layer_top > pT, p_layer_top, pT)
    lower = xr.where(p_layer_bot < pB, p_layer_bot, pB)

    overlap = xr.where(lower > upper, lower - upper, 0.0)

    return (ds_sub["O3"] * overlap * factor).sum(dim="lev")

def get_o3_series(file_path):
    """
    Extract 60–90°N area-weighted mean 30–70 hPa partial ozone column.
    """
    ds = xr.open_dataset(file_path)

    try:
        ds_sub = ds.isel(time=_get_time_mask(ds))

        o3_col = calc_parc_o3_hybrid(ds_sub, 30.0, 70.0)
        o3_nh = o3_col.sel(lat=slice(60.0, 90.0))

        weights = np.cos(np.deg2rad(o3_nh.lat))
        o3_mean = o3_nh.weighted(weights).mean(dim=["lat", "lon"]).values

    finally:
        ds.close()

    return o3_mean

# ============================================================
# 5. Climatology loader
# ============================================================
def load_o3_climatology_from_cache(clim_path):
    """
    Load the same O3 climatology cache used in the previous EGU poster figure.
    """
    if not os.path.exists(clim_path):
        raise FileNotFoundError(f"Missing climatology cache: {clim_path}")

    clim = xr.open_dataarray(clim_path)
    clim.load()
    return clim

# ============================================================
# 6. 绘图函数：只画 O3 partial column
# ============================================================
def plot_o3_partial_column_figure(
    data_dict,
    ref_series,
    clim_data,
    best_members,
    worst_members,
    save_prefix,
    title="Hindcast Ozone Column Evolution",
    ylabel="Partial ozone column, 30–70 hPa (DU)",
    ylim=(60, 140),
):
    all_series = list(data_dict.values())
    if not all_series:
        raise ValueError("No hindcast member data found. Please check Hindcast O3 directory and filename parsing.")

    # ------------------------------------------------------------
    # Ensemble statistics
    # ------------------------------------------------------------
    ens_mean = np.nanmean(all_series, axis=0)
    ens_std = np.nanstd(all_series, axis=0)
    x_days = np.arange(len(ens_mean))

    best_series = [data_dict[m] for m in best_members if m in data_dict]
    best_mean = np.nanmean(best_series, axis=0) if best_series else np.full_like(ens_mean, np.nan)

    worst_series = [data_dict[m] for m in worst_members if m in data_dict]
    worst_mean = np.nanmean(worst_series, axis=0) if worst_series else np.full_like(ens_mean, np.nan)

    # ------------------------------------------------------------
    # Figure
    # ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)

    # ------------------------------------------------------------
    # 1. Individual members
    # ------------------------------------------------------------
    cmap = plt.cm.get_cmap("tab20", len(data_dict))

    for i, (mid, series) in enumerate(data_dict.items()):
        ax.plot(
            x_days,
            series,
            color=cmap(i),
            lw=MEMBER_LW,
            alpha=MEMBER_ALPHA,
            zorder=2,
        )

    # ------------------------------------------------------------
    # 2. Ensemble mean ±1σ
    # ------------------------------------------------------------
    ax.fill_between(
        x_days,
        ens_mean - ens_std,
        ens_mean + ens_std,
        color="limegreen",
        alpha=SIGMA_ALPHA,
        zorder=3,
        label="Ens Mean ±1σ",
    )

    ax.plot(
        x_days,
        ens_mean,
        color="limegreen",
        lw=MEAN_LW,
        label="All Ens Mean",
        zorder=5,
    )

    ax.plot(
        x_days,
        best_mean,
        color="dodgerblue",
        lw=MEAN_LW,
        label=f"Best {COMPARE_N} Mean",
        zorder=6,
    )

    ax.plot(
        x_days,
        worst_mean,
        color="crimson",
        lw=MEAN_LW,
        label=f"Worst {COMPARE_N} Mean",
        zorder=6,
    )

    # ------------------------------------------------------------
    # 3. Target-year reference line
    #    不再出现 BWCN 字样
    # ------------------------------------------------------------
    if ref_series is not None:
        min_len = min(len(x_days), len(ref_series))
        ax.plot(
            x_days[:min_len],
            ref_series[:min_len],
            color="black",
            lw=REF_LW,
            label="Target year",
            zorder=10,
        )

    # ------------------------------------------------------------
    # 4. Climatology line
    #    与上一张 poster 图保持同一 cache 数据源
    # ------------------------------------------------------------
    clim_dim = list(clim_data.dims)[0]
    nclim = clim_data.sizes[clim_dim]
    clim_end = min(len(x_days), nclim)

    ax.plot(
        np.arange(clim_end),
        clim_data.isel({clim_dim: slice(0, clim_end)}).values,
        color="black",
        linestyle=":",
        lw=CLIM_LW,
        label="Climatology",
        zorder=9,
    )

    # ------------------------------------------------------------
    # 5. Axis / style
    # ------------------------------------------------------------
    ax.set_xlim(0, x_days[-1])
    ax.set_ylim(*ylim)

    ax.set_xticks([0, 31, 60, 91, 121])
    ax.set_xticklabels(
        ["Jan 1", "Feb 1", "Mar 1", "Apr 1", "May 1"],
        fontsize=TICK_FONTSIZE,
    )

    ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)

    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE)

    ax.grid(True, linestyle=":", alpha=0.6)

    ax.legend(
        loc="best",
        frameon=True,
        edgecolor="none",
        facecolor="white",
        framealpha=0.85,
        fontsize=LEGEND_FONTSIZE,
    )

    # ------------------------------------------------------------
    # 6. Save PDF / TIF / PNG
    # ------------------------------------------------------------
    base_filepath = os.path.join(PLOT_OUT, save_prefix)

    fig.savefig(f"{base_filepath}.pdf", bbox_inches="tight")

    fig.savefig(
        f"{base_filepath}.tif",
        dpi=500,
        format="tiff",
        bbox_inches="tight",
        pil_kwargs={"compression": "tiff_lzw"},
    )

    fig.savefig(
        f"{base_filepath}.png",
        dpi=300,
        bbox_inches="tight",
    )

    # ------------------------------------------------------------
    # 7. Notebook preview
    # ------------------------------------------------------------
    plt.show()

    print("[SUCCESS] Poster O3 figure saved:")
    print(f"  PDF: {base_filepath}.pdf")
    print(f"  TIF: {base_filepath}.tif")
    print(f"  PNG: {base_filepath}.png")

    return fig, ax

# ============================================================
# 7. 主运行流程：只处理 O3
# ============================================================
print("\n================ PLOT O3 PARTIAL COLUMN FOR EGU POSTER =================\n")

# ------------------------------------------------------------
# 7.1 Read RMSE ranking
# ------------------------------------------------------------
df = pd.read_csv(CSV_IN, dtype={"Member": str})
top_n = min(COMPARE_N, len(df))

best_members = set(df.head(top_n)["Member"].tolist())
worst_members = set(df.tail(top_n)["Member"].tolist())

print(f"[INFO] Best members:  {sorted(best_members)}")
print(f"[INFO] Worst members: {sorted(worst_members)}")

# ------------------------------------------------------------
# 7.2 Extract target-year reference
# ------------------------------------------------------------
try:
    ref_series = get_o3_series(REF_O3_FILE)
    print(f"[INFO] Target-year reference loaded: {REF_O3_FILE}")
except Exception as e:
    print(f"[WARNING] Failed to extract target-year reference O3: {e}")
    ref_series = None

# ------------------------------------------------------------
# 7.3 Load climatology from the same cache as previous poster figure
# ------------------------------------------------------------
clim_o3 = load_o3_climatology_from_cache(CLIM_O3_FILE)
print(f"[INFO] Climatology loaded from cache: {CLIM_O3_FILE}")

# ------------------------------------------------------------
# 7.4 Extract hindcast members
# ------------------------------------------------------------
hind_o3_dir = os.path.join(HINDCAST_BASE, "O3")
files = sorted(glob.glob(os.path.join(hind_o3_dir, "*.nc")))

if not files:
    raise FileNotFoundError(f"No O3 hindcast files found in: {hind_o3_dir}")

data_dict = {}

for f in files:
    fname = os.path.basename(f)

    try:
        # 保持你原代码的 member ID 解析逻辑
        mid = fname.split(".")[5].zfill(3)
    except Exception:
        print(f"[WARNING] Cannot parse member ID from filename, skipped: {fname}")
        continue

    try:
        data_dict[mid] = get_o3_series(f)
        print(f"[INFO] Loaded member {mid}: {fname}")
    except Exception as e:
        print(f"[WARNING] Failed to read {fname}: {e}")

# ------------------------------------------------------------
# 7.5 Plot and save
# ------------------------------------------------------------
fig, ax = plot_o3_partial_column_figure(
    data_dict=data_dict,
    ref_series=ref_series,
    clim_data=clim_o3,
    best_members=best_members,
    worst_members=worst_members,
    save_prefix=OUT_PREFIX,
    ylabel="Partial ozone column, 30–70 hPa (DU)",
    ylim=O3_YLIM,
)

# ------------------------------------------------------------
# 7.6 Optional cleanup
# ------------------------------------------------------------
plt.close(fig)
gc.collect()

print("\nPoster O3 partial-column figure successfully generated and displayed above.")

这里有的组数量不足是因为FWD是nan，因为模拟很多到六月就结束了，压根没有来得及持续7天以上的东风啥的。

In [ ]:
# ============================================================
# SINGLE PAIR SCATTER PLOT (POSTER VERSION)
# INT-3D vs Clim-3D
# Target experiment:
#   0008-02  vs  0008-02_NOCOUPL
#
# Save to:
#   /home/weiji/restart_exam/code/20260424EGUPOSTER
#
# Outputs:
#   - TIF (500 dpi)
#   - PDF (vector)
#   - PNG
#   - plt.show()
# ============================================================

import os
import re
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import pearsonr

# ============================================================
# Paths
# ============================================================

HINDCAST_ROOT = "/mnt/soclim0/public_data/weiji/Hindcast"
BWCN_FWD_TXT = "/mnt/soclim0/public_data/weiji/BWCN/BWCN_final_warming_date.txt"

CACHE_ROOT = "/home/weiji/restart_exam/code/20260415egu/plots/hindcast/O3/cache"
OUT_DIR = "/home/weiji/restart_exam/code/20260424EGUPOSTER"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# Target pair
# ============================================================

EXP_INT3D = "0008-02"
EXP_CLIM3D = "0008-02_NOCOUPL"

# ============================================================
# Figure / output controls
# ============================================================

SHOW_FIG = True
SAVE_DPI = 500
FIGSIZE = (7.0, 5.8)

USE_MANUAL_AXIS_LIMITS = False
MANUAL_XLIM = (60, 150)
MANUAL_YLIM = (60, 170)

X_PAD = 4.0
Y_PAD = 3.0

APPLY_O3_5D = True

# 是否在图上加一个小的上下文标签（推荐保留）
ADD_CONTEXT_BOX = True

# ============================================================
# Style
# ============================================================

INT3D_COLOR = "#d62728"   # red
CLIM3D_COLOR = "#1f77b4"  # blue

INT3D_ALPHA = 0.85
CLIM3D_ALPHA = 0.85
POINT_SIZE = 42

INT3D_BAND_ALPHA = 0.14
CLIM3D_BAND_ALPHA = 0.12
MEAN_LINEWIDTH = 1.5

REF_STAR_COLOR = "black"
REF_STAR_SIZE = 180
REF_STAR_EDGEWIDTH = 0.8

GRID_ALPHA = 0.45
LABEL_FONTSIZE = 12
TICK_FONTSIZE = 10
TEXT_FONTSIZE = 9.2
LEGEND_FONTSIZE = 8.2
CONTEXT_FONTSIZE = 10

# ============================================================
# Helpers
# ============================================================

def doy_to_mmdd(x, pos=None):
    if np.isnan(x) or x < 1 or x > 365:
        return ""
    base_date = datetime(2001, 1, 1)
    target_date = base_date + timedelta(days=int(round(x)) - 1)
    return target_date.strftime("%m-%d")

def load_fwd_txt(txt_path):
    fwd_dict = {}
    if not os.path.exists(txt_path):
        print(f"Warning: missing FWD txt -> {txt_path}")
        return fwd_dict

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if (not line) or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 2:
                year = int(parts[0])
                doy_str = parts[1]
                if doy_str.lower() != "nan":
                    fwd_dict[year] = float(doy_str)
    return fwd_dict

BWCN_FWD_DICT = load_fwd_txt(BWCN_FWD_TXT)

def get_init_offset_from_exp_name(exp_name):
    """
    metrics.txt 里的 FWD_DOY 是相对起报日，这里转成全年绝对 DOY
    """
    m = re.match(r"^\d{4}-(\d{2})", exp_name)
    if m is None:
        raise ValueError(f"Cannot infer init month from exp_name: {exp_name}")

    mm = int(m.group(1))
    month_start_doy = {
        1: 0,
        2: 31,
        3: 59,
        4: 90,
    }
    if mm not in month_start_doy:
        raise ValueError(f"Unsupported init month in exp_name: {exp_name}")
    return month_start_doy[mm]

def parse_exp_context(exp_name):
    """
    把 0008-02 这种名字转成更适合图上的说明
    """
    m = re.match(r"^(\d{4})-(\d{2})", exp_name)
    if m is None:
        raise ValueError(f"Cannot parse exp name: {exp_name}")

    year_str = m.group(1)
    init_month = int(m.group(2))

    month_name = {
        1: "Jan",
        2: "Feb",
        3: "Mar",
        4: "Apr",
    }.get(init_month, f"Month{init_month:02d}")

    return {
        "event_year": year_str,
        "init_month": init_month,
        "init_label": f"Initialized 1 {month_name}",
        "context_label": f"Event year {year_str}\nInitialized 1 {month_name}",
    }

def load_metrics_txt(exp_name):
    metrics_txt = os.path.join(HINDCAST_ROOT, exp_name, f"{exp_name}_metrics.txt")
    if not os.path.exists(metrics_txt):
        raise FileNotFoundError(f"Missing metrics txt -> {metrics_txt}")

    df = pd.read_csv(metrics_txt, sep=r"\s+", engine="python")

    required = ["Member", "FWD_DOY", "O3_MA_Min_Val"]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"{metrics_txt} missing column: {c}")

    df["Member"] = pd.to_numeric(df["Member"], errors="coerce")
    df["FWD_DOY"] = pd.to_numeric(df["FWD_DOY"], errors="coerce")
    df["O3_MA_Min_Val"] = pd.to_numeric(df["O3_MA_Min_Val"], errors="coerce")

    # 与你原始逻辑保持一致：只画 FWD 和 O3 都有效的成员
    df = df[np.isfinite(df["FWD_DOY"]) & np.isfinite(df["O3_MA_Min_Val"])].copy()
    df = df.sort_values("Member").reset_index(drop=True)

    df["FWD_DOY_REL"] = df["FWD_DOY"].astype(float)
    df["FWD_DOY_ABS"] = df["FWD_DOY_REL"] + get_init_offset_from_exp_name(exp_name)

    return df

_BWCN_REF_O3MIN_CACHE = {}

def get_bwcn_reference_o3min_for_year(year_str, apply_o3_5d=True):
    """
    从 BWCN O3 cache 里计算参考年的 3–4 月 O3 minimum
    """
    cache_key = (year_str, bool(apply_o3_5d))
    if cache_key in _BWCN_REF_O3MIN_CACHE:
        return _BWCN_REF_O3MIN_CACHE[cache_key]

    o3_nc = os.path.join(CACHE_ROOT, "O3", f"O3_BWCN_{year_str}.nc")
    if not os.path.exists(o3_nc):
        print(f"Warning: missing BWCN O3 cache -> {o3_nc}")
        _BWCN_REF_O3MIN_CACHE[cache_key] = np.nan
        return np.nan

    da = xr.open_dataarray(o3_nc).load().sortby("time")

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    mask = da.time.dt.month.isin([3, 4])
    seg = da.where(mask, drop=True)

    if seg.size == 0 or int(seg.count().values) == 0:
        val = np.nan
    else:
        val = float(seg.min().values)

    da.close()
    _BWCN_REF_O3MIN_CACHE[cache_key] = val
    return val

def get_reference_star_for_year(year_str):
    """
    BWCN 参考星号位置:
      x_ref = BWCN reference FWD
      y_ref = BWCN reference O3 minimum
    """
    year_int = int(year_str)
    x_ref = BWCN_FWD_DICT.get(year_int, np.nan)
    y_ref = get_bwcn_reference_o3min_for_year(year_str, apply_o3_5d=APPLY_O3_5D)
    return x_ref, y_ref

def compute_pair_axis_limits(df1, df2, ref_xy=None):
    if USE_MANUAL_AXIS_LIMITS:
        return MANUAL_XLIM, MANUAL_YLIM

    all_x = []
    all_y = []

    for df in [df1, df2]:
        if df is not None and len(df) > 0:
            all_x.extend(df["FWD_DOY_ABS"].values.astype(float))
            all_y.extend(df["O3_MA_Min_Val"].values.astype(float))

    if ref_xy is not None:
        x_ref, y_ref = ref_xy
        if np.isfinite(x_ref) and np.isfinite(y_ref):
            all_x.append(float(x_ref))
            all_y.append(float(y_ref))

    all_x = np.asarray(all_x, dtype=float)
    all_y = np.asarray(all_y, dtype=float)

    if all_x.size == 0 or all_y.size == 0:
        return (60, 150), (60, 170)

    xmin = np.nanmin(all_x) - X_PAD
    xmax = np.nanmax(all_x) + X_PAD
    ymin = np.nanmin(all_y) - Y_PAD
    ymax = np.nanmax(all_y) + Y_PAD

    return (xmin, xmax), (ymin, ymax)

def stats_text(df):
    if df is None or len(df) == 0:
        return "N = 0"

    x = df["FWD_DOY_ABS"].values.astype(float)
    y = df["O3_MA_Min_Val"].values.astype(float)
    n = len(x)

    if n >= 3:
        try:
            r, p = pearsonr(x, y)
            return f"r = {r:.3f}\np = {p:.2e}\nN = {n}"
        except Exception:
            return f"N = {n}"
    return f"N = {n}"

# ============================================================
# Plotting
# ============================================================

def draw_pair_scatter(ax, df_int3d, df_clim3d, xlim, ylim, ref_xy=None, context_text=None):
    ax.grid(True, linestyle=":", alpha=GRID_ALPHA)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # ---------------- INT-3D ----------------
    if df_int3d is not None and len(df_int3d) > 0:
        x_i = df_int3d["FWD_DOY_ABS"].values.astype(float)
        y_i = df_int3d["O3_MA_Min_Val"].values.astype(float)

        ax.scatter(
            x_i, y_i,
            s=POINT_SIZE,
            color=INT3D_COLOR,
            alpha=INT3D_ALPHA,
            edgecolors="none",
            zorder=3
        )

        mean_i = np.nanmean(x_i)
        std_i = np.nanstd(x_i, ddof=0)

        ax.axvspan(
            mean_i - std_i, mean_i + std_i,
            color=INT3D_COLOR,
            alpha=INT3D_BAND_ALPHA,
            zorder=0
        )
        ax.axvline(
            mean_i,
            color=INT3D_COLOR,
            linestyle="-",
            linewidth=MEAN_LINEWIDTH,
            alpha=0.95,
            zorder=1
        )
    else:
        mean_i = np.nan

    # ---------------- Clim-3D ----------------
    if df_clim3d is not None and len(df_clim3d) > 0:
        x_c = df_clim3d["FWD_DOY_ABS"].values.astype(float)
        y_c = df_clim3d["O3_MA_Min_Val"].values.astype(float)

        ax.scatter(
            x_c, y_c,
            s=POINT_SIZE,
            color=CLIM3D_COLOR,
            alpha=CLIM3D_ALPHA,
            edgecolors="none",
            zorder=3
        )

        mean_c = np.nanmean(x_c)
        std_c = np.nanstd(x_c, ddof=0)

        ax.axvspan(
            mean_c - std_c, mean_c + std_c,
            color=CLIM3D_COLOR,
            alpha=CLIM3D_BAND_ALPHA,
            zorder=0
        )
        ax.axvline(
            mean_c,
            color=CLIM3D_COLOR,
            linestyle="-",
            linewidth=MEAN_LINEWIDTH,
            alpha=0.95,
            zorder=1
        )
    else:
        mean_c = np.nan

    # ---------------- BWCN reference star ----------------
    x_ref, y_ref = (np.nan, np.nan) if ref_xy is None else ref_xy
    if np.isfinite(x_ref) and np.isfinite(y_ref):
        ax.scatter(
            x_ref, y_ref,
            s=REF_STAR_SIZE,
            marker="*",
            color=REF_STAR_COLOR,
            edgecolors="white",
            linewidths=REF_STAR_EDGEWIDTH,
            zorder=6
        )

    # ---------------- text boxes ----------------
    txt_i = stats_text(df_int3d)
    txt_c = stats_text(df_clim3d)

    ax.text(
        0.03, 0.96, "INT-3D\n" + txt_i,
        transform=ax.transAxes,
        va="top", ha="left",
        fontsize=TEXT_FONTSIZE,
        color=INT3D_COLOR,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.42, edgecolor=INT3D_COLOR)
    )

    ax.text(
        0.97, 0.96, "Clim-3D\n" + txt_c,
        transform=ax.transAxes,
        va="top", ha="right",
        fontsize=TEXT_FONTSIZE,
        color=CLIM3D_COLOR,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.42, edgecolor=CLIM3D_COLOR)
    )

    # ---------------- context box ----------------
    if ADD_CONTEXT_BOX and context_text is not None:
        ax.text(
            0.50, 0.96, context_text,
            transform=ax.transAxes,
            va="top", ha="center",
            fontsize=CONTEXT_FONTSIZE,
            color="black",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.38, edgecolor="0.5")
        )

    # ---------------- legend ----------------
    handles = [
        Line2D([0], [0], marker="o", color="none",
               markerfacecolor=INT3D_COLOR, markeredgecolor="none",
               markersize=6.5, alpha=INT3D_ALPHA),
        Line2D([0], [0], marker="o", color="none",
               markerfacecolor=CLIM3D_COLOR, markeredgecolor="none",
               markersize=6.5, alpha=CLIM3D_ALPHA),
        Line2D([0], [0], color=INT3D_COLOR, lw=MEAN_LINEWIDTH, ls="-"),
        Patch(facecolor=INT3D_COLOR, edgecolor="none", alpha=INT3D_BAND_ALPHA),
        Line2D([0], [0], color=CLIM3D_COLOR, lw=MEAN_LINEWIDTH, ls="-"),
        Patch(facecolor=CLIM3D_COLOR, edgecolor="none", alpha=CLIM3D_BAND_ALPHA),
    ]
    labels = [
        "INT-3D members",
        "Clim-3D members",
        f"INT-3D mean FWD: {doy_to_mmdd(mean_i)}" if np.isfinite(mean_i) else "INT-3D mean FWD: NA",
        r"INT-3D FWD $\pm 1\sigma$",
        f"Clim-3D mean FWD: {doy_to_mmdd(mean_c)}" if np.isfinite(mean_c) else "Clim-3D mean FWD: NA",
        r"Clim-3D FWD $\pm 1\sigma$",
    ]

    if np.isfinite(x_ref) and np.isfinite(y_ref):
        handles.append(
            Line2D([0], [0], marker="*", color="none",
                   markerfacecolor=REF_STAR_COLOR, markeredgecolor="white",
                   markeredgewidth=0.8, markersize=11, linewidth=0)
        )
        labels.append("BWCN reference")

    ax.legend(
        handles, labels,
        loc="lower left",
        fontsize=LEGEND_FONTSIZE,
        frameon=False,
        handletextpad=0.45,
        borderpad=0.2,
        labelspacing=0.3
    )

    ax.set_xlabel("Final warming date", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(r"Mar–Apr minimum O$_3$ column (30–70 hPa, DU)", fontsize=LABEL_FONTSIZE)
    ax.xaxis.set_major_formatter(FuncFormatter(doy_to_mmdd))
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)

    for tick in ax.get_xticklabels():
        tick.set_rotation(45)

# ============================================================
# Main
# ============================================================

meta = parse_exp_context(EXP_INT3D)

df_int3d = load_metrics_txt(EXP_INT3D)
df_clim3d = load_metrics_txt(EXP_CLIM3D)

year = meta["event_year"]
ref_xy = get_reference_star_for_year(year)

xlim, ylim = compute_pair_axis_limits(df_int3d, df_clim3d, ref_xy=ref_xy)

fig, ax = plt.subplots(figsize=FIGSIZE)

draw_pair_scatter(
    ax=ax,
    df_int3d=df_int3d,
    df_clim3d=df_clim3d,
    xlim=xlim,
    ylim=ylim,
    ref_xy=ref_xy,
    context_text=meta["context_label"]
)

# 不加总标题 / suptitle，避免重复

base = f"FWD_vs_O3min_Event{meta['event_year']}_Init1Feb_INT3D_vs_Clim3D"

png_path = os.path.join(OUT_DIR, base + ".png")
tif_path = os.path.join(OUT_DIR, base + ".tif")
pdf_path = os.path.join(OUT_DIR, base + ".pdf")

fig.savefig(png_path, dpi=SAVE_DPI, bbox_inches="tight")
fig.savefig(tif_path, dpi=SAVE_DPI, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")

print("Saved files:")
print("  PNG :", png_path)
print("  TIF :", tif_path)
print("  PDF :", pdf_path)
print("  xlim =", xlim)
print("  ylim =", ylim)
print("  BWCN ref star =", ref_xy)

plt.show()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import pearsonr

# ============================================================
# Paths & Config
# ============================================================
FW_TXT_BWCN    = "/mnt/soclim0/public_data/weiji/BWCN/BWCN_final_warming_date.txt"
FW_TXT_B2000   = "/mnt/soclim0/public_data/weiji/B2000WCN001002/B2000WCN_final_warming_date.txt"
FW_TXT_NOCOUPL = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/B2000WCN_NOCOUPL_final_warming_date.txt"
FW_TXT_MERRA2  = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/MERRA2_final_warming_date.txt"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"

# >>> 改到 EGU POSTER 输出目录
OUT_DIR = "/home/weiji/restart_exam/code/20260424EGUPOSTER"
os.makedirs(OUT_DIR, exist_ok=True)

BRIDGE_YEAR = 104
APPLY_O3_5D = True

# 输出分辨率
SAVE_DPI = 500
SHOW_FIG = True

# 画布
FIGSIZE = (16.2, 5.4)

BG_COLOR = "0.35"
LOW25_COLOR = "red"
BG_ALPHA = 0.75
LOW25_ALPHA = 0.95
BG_SIZE = 34
LOW25_SIZE = 42

MARK_EXTREMES = True
EXTREME_TOPN = 5
EXTREME_BY = "y_raw"
EXTREME_ASCENDING = True
EXTREME_SIZE = 120
EXTREME_LINEWIDTH = 1.6
EXTREME_COLORS = ['#8B0000', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

USE_MANUAL_AXIS_LIMITS = False
MANUAL_XLIM = (60, 150)
MANUAL_YLIM = (60, 170)

# climatology / FWD band styles
CLIM_COLOR = "black"
CLIM_LINESTYLE = "-"
CLIM_LINEWIDTH = 1.6
CLIM_ALPHA = 0.95

FWD_MEAN_COLOR = "#1f77b4"
FWD_BAND_COLOR = "#1f77b4"
FWD_MEAN_LINESTYLE = "--"
FWD_MEAN_LINEWIDTH = 1.2
FWD_BAND_ALPHA = 0.10

# 字体
PANEL_TITLE_SIZE = 13
AXIS_LABEL_SIZE = 12
TICK_SIZE = 10
TEXT_SIZE = 9.2
LEGEND_SIZE = 5.2

# ============================================================
# Helpers
# ============================================================
def doy_to_mmdd(x, pos=None):
    if np.isnan(x) or x < 1 or x > 365:
        return ""
    base_date = datetime(2001, 1, 1)
    target_date = base_date + timedelta(days=int(round(x)) - 1)
    return target_date.strftime("%m-%d")


def load_fwd_data(txt_path):
    fwd_dict = {}
    if not os.path.exists(txt_path):
        print(f"Warning: FWD file not found -> {txt_path}")
        return fwd_dict

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("#") or not line:
                continue
            parts = line.split()
            if len(parts) >= 2:
                year = int(parts[0])
                doy_str = parts[1]
                if doy_str.lower() != "nan":
                    fwd_dict[year] = float(doy_str)
    return fwd_dict


def assert_daily_unique(da, name=""):
    da = da.sortby("time")
    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)
    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError(f"{name}: duplicated daily timestamps detected.")
    return da


def drop_feb29(da):
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)


def build_mmdd_order():
    month_days = [
        (1, 31), (2, 28), (3, 31), (4, 30), (5, 31), (6, 30),
        (7, 31), (8, 31), (9, 30), (10, 31), (11, 30), (12, 31)
    ]
    out = []
    for m, nd in month_days:
        for d in range(1, nd + 1):
            out.append(f"{m:02d}-{d:02d}")
    return out


MMDD_ORDER = build_mmdd_order()
MMDD_TO_DOY = {mmdd: i + 1 for i, mmdd in enumerate(MMDD_ORDER)}


def get_ma_min_o3_series(o3_nc, apply_o3_5d=True):
    if not os.path.exists(o3_nc):
        print(f"Warning: O3 file not found -> {o3_nc}")
        return {}

    da = xr.open_dataarray(o3_nc)
    da = assert_daily_unique(da, name="O3").sortby("time")

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    years = np.unique(da.time.dt.year.values.astype(int))
    o3_dict = {}

    for yr in years:
        mask = (da.time.dt.year == yr) & (da.time.dt.month.isin([3, 4]))
        seg = da.where(mask, drop=True)
        n_valid = int(seg.count().values) if seg.size > 0 else 0
        if n_valid >= 58:
            o3_dict[int(yr)] = float(seg.min().values)

    da.close()
    return o3_dict


def get_low25_years(txt_path):
    if txt_path and os.path.exists(txt_path):
        return set(np.loadtxt(txt_path, dtype=int).reshape(-1))
    return set()


def load_o3_daily_climatology_curve(o3_nc, apply_o3_5d=True, drop_years=None):
    """
    返回 climatology curve:
      x_clim: 1..365 (DOY, noleap)
      y_clim: daily climatological partial O3 column
      n_years: 实际参与 climatology 的有效年份数
    """
    if not os.path.exists(o3_nc):
        print(f"Warning: O3 file not found -> {o3_nc}")
        return None

    da = xr.open_dataarray(o3_nc)
    da = assert_daily_unique(da, name="O3_daily").sortby("time")
    da = drop_feb29(da)

    if drop_years:
        da = da.sel(time=~da.time.dt.year.isin(list(drop_years)))

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    years = np.unique(da.time.dt.year.values.astype(int))
    valid_years = []
    for yr in years:
        seg = da.sel(time=da.time.dt.year == yr)
        n_valid = int(seg.count().values) if seg.size > 0 else 0
        if n_valid >= 300:
            valid_years.append(int(yr))

    mmdd = np.array([f"{m:02d}-{d:02d}" for m, d in zip(
        da.time.dt.month.values.astype(int),
        da.time.dt.day.values.astype(int)
    )])
    da = da.assign_coords(mmdd=("time", mmdd))

    clim = da.groupby("mmdd").mean("time")
    clim = clim.sel(mmdd=MMDD_ORDER)

    x_clim = np.arange(1, 366, dtype=float)
    y_clim = np.asarray(clim.values, dtype=float)

    da.close()
    return {
        "x": x_clim,
        "y": y_clim,
        "n_years": len(valid_years),
    }


def average_climatology_curves(curves):
    curves = [c for c in curves if c is not None]
    if len(curves) == 0:
        return None
    if len(curves) == 1:
        return curves[0]

    x = curves[0]["x"]
    weights = np.array([c.get("n_years", 1) for c in curves], dtype=float)
    y_stack = np.vstack([c["y"] for c in curves])

    y_avg = np.average(y_stack, axis=0, weights=weights)

    return {
        "x": x,
        "y": y_avg,
        "n_years": int(np.sum(weights)),
    }


def build_scatter_df(fwd_txt, o3_nc, low25_txt, part_name, is_bridge_case=False):
    fwd_data = load_fwd_data(fwd_txt)
    o3_data = get_ma_min_o3_series(o3_nc, apply_o3_5d=APPLY_O3_5D)
    low25_set = get_low25_years(low25_txt)

    common_years = sorted(list(set(fwd_data.keys()).intersection(set(o3_data.keys()))))

    if is_bridge_case and BRIDGE_YEAR in common_years:
        common_years.remove(BRIDGE_YEAR)

    x_list, y_list, low25_mask, year_list, part_list = [], [], [], [], []
    for y in common_years:
        x_list.append(fwd_data[y])
        y_list.append(o3_data[y])
        low25_mask.append(y in low25_set)
        year_list.append(y)
        part_list.append(part_name)

    return {
        "x": np.array(x_list, dtype=float),
        "y": np.array(y_list, dtype=float),
        "y_raw": np.array(y_list, dtype=float),
        "year": np.array(year_list, dtype=int),
        "part": np.array(part_list, dtype=object),
        "is_low25": np.array(low25_mask, dtype=bool)
    }


def combine_dfs(df1, df2):
    if df1 is None or df1["x"].size == 0:
        return df2
    if df2 is None or df2["x"].size == 0:
        return df1
    return {
        "x": np.concatenate([df1["x"], df2["x"]]),
        "y": np.concatenate([df1["y"], df2["y"]]),
        "y_raw": np.concatenate([df1["y_raw"], df2["y_raw"]]),
        "year": np.concatenate([df1["year"], df2["year"]]),
        "part": np.concatenate([df1["part"], df2["part"]]),
        "is_low25": np.concatenate([df1["is_low25"], df2["is_low25"]])
    }


def select_panel_extremes(df, topn=5, extreme_by="y_raw", ascending=True):
    if df is None or len(df["year"]) == 0:
        return np.array([], dtype=int)

    vals = np.asarray(df[extreme_by], dtype=float)
    valid_idx = np.where(np.isfinite(vals))[0]
    if valid_idx.size == 0:
        return np.array([], dtype=int)

    order = np.argsort(vals[valid_idx])
    if not ascending:
        order = order[::-1]

    keep = valid_idx[order[:min(topn, len(order))]]
    return keep


def overlay_panel_extremes(ax, df, idx_extreme):
    if df is None or len(idx_extreme) == 0:
        return []

    legend_entries = []

    for i, idx in enumerate(idx_extreme):
        c = EXTREME_COLORS[i % len(EXTREME_COLORS)]
        x = float(df["x"][idx])
        y = float(df["y"][idx])
        part = str(df["part"][idx])
        year = int(df["year"][idx])

        if part == "BWCN":
            label = f"BWCN-{year:04d}"
        else:
            label = f"{year:04d}"

        legend_entries.append((i, label, c, part))

        if part == "BWCN":
            ax.scatter(
                x, y,
                s=EXTREME_SIZE,
                facecolors="none",
                edgecolors=c,
                linewidths=EXTREME_LINEWIDTH,
                zorder=10
            )
        else:
            ax.scatter(
                x, y,
                s=EXTREME_SIZE,
                color=c,
                edgecolors="k",
                linewidths=0.8,
                zorder=10
            )

    return legend_entries


def compute_global_axis_limits(dfs, clim_curves=None, use_manual=False,
                               manual_xlim=None, manual_ylim=None,
                               xpad=5.0, ypad=3.0):
    if use_manual:
        return manual_xlim, manual_ylim

    all_x = []
    all_y = []

    for df in dfs:
        if df is not None and df["x"].size > 0:
            all_x.extend(df["x"])
            all_y.extend(df["y"])

    if clim_curves is not None:
        for c in clim_curves:
            if c is not None:
                all_x.extend(c["x"])
                all_y.extend(c["y"])

    all_x = np.asarray(all_x, dtype=float)
    all_y = np.asarray(all_y, dtype=float)

    if all_x.size == 0 or all_y.size == 0:
        return (60, 150), (60, 170)

    xmin, xmax = np.nanmin(all_x), np.nanmax(all_x)
    ymin, ymax = np.nanmin(all_y), np.nanmax(all_y)

    return (xmin - xpad, xmax + xpad), (ymin - ypad, ymax + ypad)


# ============================================================
# Plotting
# ============================================================
def draw_one_panel(ax, df, clim_curve, panel_title, xlim, ylim,
                   show_left_ylabel=True, show_right_ylabel=False):
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # 右侧副轴
    ax2 = ax.twinx()
    ax2.set_ylim(ylim)

    # 只在最右边保留右侧副轴标签与刻度
    if show_right_ylabel:
        ax2.set_ylabel(r"O$_3$ climatology (DU)", color=CLIM_COLOR, fontsize=AXIS_LABEL_SIZE)
        ax2.tick_params(axis='y', colors=CLIM_COLOR, labelsize=TICK_SIZE)
    else:
        ax2.set_ylabel("")
        ax2.tick_params(axis='y', right=False, labelright=False)
        ax2.spines["right"].set_visible(False)

    if df is None or len(df["x"]) == 0:
        ax.set_title(panel_title, fontweight="bold", fontsize=PANEL_TITLE_SIZE)
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlabel("Final warming date", fontsize=AXIS_LABEL_SIZE)
        if show_left_ylabel:
            ax.set_ylabel(r"Mar–Apr minimum O$_3$ (DU)", fontsize=AXIS_LABEL_SIZE)
        else:
            ax.set_ylabel("")
            ax.tick_params(axis="y", left=False, labelleft=False)
            ax.spines["left"].set_visible(False)
        ax.xaxis.set_major_formatter(FuncFormatter(doy_to_mmdd))
        ax.tick_params(axis="x", labelsize=TICK_SIZE)
        return ax2

    low_mask = df["is_low25"]
    bg_mask = ~low_mask

    # scatter points
    if np.any(bg_mask):
        ax.scatter(
            df["x"][bg_mask], df["y"][bg_mask],
            s=BG_SIZE, color=BG_COLOR, alpha=BG_ALPHA,
            edgecolors="none", zorder=2
        )

    part_arr = np.asarray(df["part"]).astype(str)
    bwcn_mask = part_arr == "BWCN"
    other_mask = ~bwcn_mask

    # low25: B2000WCN / MERRA2 / Clim-3D 用红色实心
    low_other = low_mask & other_mask
    if np.any(low_other):
        ax.scatter(
            df["x"][low_other], df["y"][low_other],
            s=LOW25_SIZE,
            color=LOW25_COLOR,
            alpha=LOW25_ALPHA,
            edgecolors="k",
            linewidths=0.4,
            zorder=5
        )

    # low25: BWCN 用红色空心圆
    low_bwcn = low_mask & bwcn_mask
    if np.any(low_bwcn):
        ax.scatter(
            df["x"][low_bwcn], df["y"][low_bwcn],
            s=LOW25_SIZE,
            facecolors="none",
            edgecolors=LOW25_COLOR,
            linewidths=1.2,
            alpha=0.95,
            zorder=6
        )

    # extreme points
    legend_entries = []
    if MARK_EXTREMES:
        idx_ext = select_panel_extremes(
            df,
            topn=EXTREME_TOPN,
            extreme_by=EXTREME_BY,
            ascending=EXTREME_ASCENDING
        )
        legend_entries = overlay_panel_extremes(ax, df, idx_ext)

    # FWD mean ± sigma
    mean_fwd = np.nanmean(df["x"])
    std_fwd  = np.nanstd(df["x"], ddof=0)
    mean_fwd_low25 = np.nanmean(df["x"][low_mask]) if np.any(low_mask) else np.nan
    std_fwd_low25 = np.nanstd(df["x"][low_mask], ddof=0) if np.any(low_mask) else np.nan

    ax.axvspan(mean_fwd - std_fwd, mean_fwd + std_fwd,
               color=FWD_BAND_COLOR, alpha=0.18, zorder=0)

    ax.axvline(mean_fwd, color=FWD_MEAN_COLOR, linestyle="-",
               linewidth=1.4, alpha=0.95, zorder=1)

    ax.axvline(mean_fwd - std_fwd, color=FWD_MEAN_COLOR, linestyle="--",
               linewidth=1.0, alpha=0.8, zorder=1)
    ax.axvline(mean_fwd + std_fwd, color=FWD_MEAN_COLOR, linestyle="--",
               linewidth=1.0, alpha=0.8, zorder=1)

    if np.isfinite(mean_fwd_low25):
        ax.axvline(mean_fwd_low25, color=LOW25_COLOR, linestyle="-",
                   linewidth=1.0, alpha=0.8, zorder=1)

    if np.isfinite(mean_fwd_low25) and np.isfinite(std_fwd_low25):
        ax.axvspan(mean_fwd_low25 - std_fwd_low25,
                   mean_fwd_low25 + std_fwd_low25,
                   color=LOW25_COLOR, alpha=0.12, zorder=0)

    # climatology curve
    if clim_curve is not None:
        ax2.plot(
            clim_curve["x"], clim_curve["y"],
            color=CLIM_COLOR, linestyle=CLIM_LINESTYLE,
            linewidth=CLIM_LINEWIDTH, alpha=CLIM_ALPHA, zorder=1
        )

    # statistics
    n_all = len(df["x"])
    if n_all >= 3:
        try:
            r_all, p_all = pearsonr(df["x"], df["y"])
            txt_all = f"ALL\nr = {r_all:.3f}\np = {p_all:.2e}\nN = {n_all}"
        except Exception:
            txt_all = f"ALL\nN = {n_all}"
    else:
        txt_all = f"ALL\nN = {n_all}"

    ax.text(
        0.04, 0.96, txt_all,
        transform=ax.transAxes, va="top", ha="left", fontsize=TEXT_SIZE, color="k",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.35, edgecolor="0.7")
    )

    n_low = int(np.sum(low_mask))
    if n_low >= 3:
        try:
            r_low, p_low = pearsonr(df["x"][low_mask], df["y"][low_mask])
            txt_low = f"LOW25\nr = {r_low:.3f}\np = {p_low:.2e}\nN = {n_low}"
        except Exception:
            txt_low = f"LOW25\nN = {n_low}"
    else:
        txt_low = f"LOW25\nN = {n_low}"

    ax.text(
        0.96, 0.96, txt_low,
        transform=ax.transAxes, va="top", ha="right", fontsize=TEXT_SIZE, color=LOW25_COLOR,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.35, edgecolor=LOW25_COLOR)
    )

    # legend
    handles = []
    labels = []

    for i, label, c, part in legend_entries:
        if part == "BWCN":
            h = Line2D(
                [0], [0],
                marker="o",
                color="none",
                markeredgecolor=c,
                markerfacecolor="none",
                markeredgewidth=1.2,
                markersize=5.0,
                linewidth=0
            )
        else:
            h = Line2D(
                [0], [0],
                marker="o",
                color="none",
                markeredgecolor="k",
                markerfacecolor=c,
                markeredgewidth=0.6,
                markersize=5.0,
                linewidth=0
            )
        handles.append(h)
        labels.append(f"Top{i+1}: {label}")

    handles.extend([
        Line2D([0], [0], color=CLIM_COLOR, lw=CLIM_LINEWIDTH, ls=CLIM_LINESTYLE),
        Line2D([0], [0], color=FWD_MEAN_COLOR, lw=1.4, ls="-"),
        Patch(facecolor=FWD_BAND_COLOR, edgecolor="none", alpha=0.18),
    ])
    labels.extend([
        "O$_3$ climatology",
        f"ALL mean FWD: {doy_to_mmdd(mean_fwd)}",
        r"ALL FWD $\pm 1\sigma$"
    ])

    if np.isfinite(mean_fwd_low25):
        handles.append(Line2D([0], [0], color=LOW25_COLOR, lw=1.4, ls="-"))
        labels.append(f"LOW25 mean FWD: {doy_to_mmdd(mean_fwd_low25)}")

    ax.legend(
        handles, labels,
        loc="lower left",
        fontsize=LEGEND_SIZE,
        frameon=False,
        handletextpad=0.35,
        borderpad=0.2,
        labelspacing=0.25
    )

    ax.set_title(panel_title, fontweight="bold", fontsize=PANEL_TITLE_SIZE)
    ax.set_xlabel("Final warming date", fontsize=AXIS_LABEL_SIZE)

    # 左轴：只保留最左边
    if show_left_ylabel:
        ax.set_ylabel(r"Mar–Apr minimum O$_3$ (DU)", fontsize=AXIS_LABEL_SIZE)
        ax.tick_params(axis="y", labelsize=TICK_SIZE)
    else:
        ax.set_ylabel("")
        ax.tick_params(axis="y", left=False, labelleft=False)
        ax.spines["left"].set_visible(False)

    ax.xaxis.set_major_formatter(FuncFormatter(doy_to_mmdd))
    ax.tick_params(axis="x", labelsize=TICK_SIZE)

    return ax2


# ============================================================
# Main Execution
# ============================================================
if __name__ == "__main__":
    print("Loading data...")

    # scatter data
    df_merra = build_scatter_df(
        FW_TXT_MERRA2,
        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
        os.path.join(DATA_BASE, "MERRA2", "low25pct_years_30_70hPa.txt"),
        part_name="MERRA2",
        is_bridge_case=False
    )

    df_bwcn = build_scatter_df(
        FW_TXT_BWCN,
        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "BWCN", "low25pct_years_30_70hPa.txt"),
        part_name="BWCN",
        is_bridge_case=False
    )

    df_b2000 = build_scatter_df(
        FW_TXT_B2000,
        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN", "low25pct_years_30_70hPa.txt"),
        part_name="B2000WCN",
        is_bridge_case=True
    )

    # INT-3D = BWCN + B2000WCN
    df_int3d = combine_dfs(df_bwcn, df_b2000)

    # Clim-3D
    df_clim3d = build_scatter_df(
        FW_TXT_NOCOUPL,
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "low25pct_years_30_70hPa.txt"),
        part_name="B2000WCN_NOCOUPL",
        is_bridge_case=True
    )

    # climatology curves
    clim_merra = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years=None
    )

    clim_bwcn = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years=None
    )

    clim_b2000 = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years={BRIDGE_YEAR}
    )

    clim_int3d = average_climatology_curves([clim_bwcn, clim_b2000])

    clim_clim3d = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years={BRIDGE_YEAR}
    )

    # global axes
    xlim, ylim = compute_global_axis_limits(
        [df_merra, df_int3d, df_clim3d],
        clim_curves=None,
        use_manual=USE_MANUAL_AXIS_LIMITS,
        manual_xlim=MANUAL_XLIM,
        manual_ylim=MANUAL_YLIM
    )

    print("Global xlim =", xlim)
    print("Global ylim =", ylim)

    print("Plotting...")

    # sharey=True 让三张图共用主 Y 轴
    fig, axes = plt.subplots(
        1, 3,
        figsize=FIGSIZE,
        sharey=True
    )
    plt.subplots_adjust(wspace=0.08)

    # 左：保留左 Y 轴；不保留右 Y 轴
    ax2_0 = draw_one_panel(
        axes[0], df_merra, clim_merra, "MERRA2", xlim, ylim,
        show_left_ylabel=True, show_right_ylabel=False
    )

    # 中：左右 Y 轴都隐藏
    ax2_1 = draw_one_panel(
        axes[1], df_int3d, clim_int3d, "INT-3D", xlim, ylim,
        show_left_ylabel=False, show_right_ylabel=False
    )

    # 右：隐藏左 Y 轴；保留右 Y 轴
    ax2_2 = draw_one_panel(
        axes[2], df_clim3d, clim_clim3d, "Clim-3D", xlim, ylim,
        show_left_ylabel=False, show_right_ylabel=True
    )

    # 所有 x tick 旋转
    for ax in axes:
        for tick in ax.get_xticklabels():
            tick.set_rotation(45)

    # 不强制 suptitle，poster 上通常可不用
    # fig.suptitle("Mar–Apr minimum O$_3$ vs Final warming date", fontsize=15, fontweight="bold", y=1.02)

    base = "Scatter_FWD_vs_O3_MERRA2_INT3D_Clim3D"

    png_path = os.path.join(OUT_DIR, base + ".png")
    tif_path = os.path.join(OUT_DIR, base + ".tif")
    pdf_path = os.path.join(OUT_DIR, base + ".pdf")

    fig.savefig(png_path, dpi=SAVE_DPI, bbox_inches="tight")
    fig.savefig(tif_path, dpi=SAVE_DPI, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")

    if SHOW_FIG:
        plt.show()
    else:
        plt.close(fig)

    print("Saved successfully:")
    print("  PNG :", png_path)
    print("  TIF :", tif_path)
    print("  PDF :", pdf_path)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
from datetime import datetime
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

# ============================================================
# Paths
# ============================================================
FILE_BWCN_EP     = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP    = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
FILE_NOCOUPL_EP  = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/EPflux_daily/EPFLUX_205yr_Daily_Series_Full.nc"
FILE_MERRA_EP    = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
OUT_DIR   = "/home/weiji/restart_exam/code/20260424EGUPOSTER"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# Main config
# ============================================================
FZ_SEASON = "DJF"
FZ_METHOD = "mean"

O3_SEASON = "MA"
O3_METHOD = "min"
APPLY_O3_5D = True

LAT_BAND = (40, 80)

BRIDGE_EXPERIMENTS = {"B2000WCN", "B2000WCN_NOCOUPL"}
BRIDGE_YEAR = 104

X_PLOT_SCALE = 1.0e3  # raw -Fz * 1000 -> 10^-3 hPa m s^-2

# 是否手动固定坐标轴
USE_MANUAL_AXIS_LIMITS = True
MANUAL_XLIM = (0, 5)
MANUAL_YLIM = (60, 140)

X_PAD = 0.2
Y_PAD = 3.0

# 极端年高亮（panel 内最低 O3 的 TopN）
MARK_EXTREMES = True
EXTREME_TOPN = 5
EXTREME_COLORS = ['#8B0000', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

# 输出
SAVE_DPI = 500
SHOW_FIG = True

# Figure style
FIGSIZE = (15.8, 5.6)
TITLE_SIZE = 13
LABEL_SIZE = 12
TICK_SIZE = 10
TEXT_SIZE = 9.0
LEGEND_SIZE = 6.2

# 点样式
GRAY_FILL = "0.20"
GRAY_EDGE = "black"
LOW25_COLOR = "red"

BASE_SIZE = 42
LOW25_SIZE = 48
EXTREME_SIZE = 130

BASE_ALPHA_FILLED = 0.48
BASE_ALPHA_HOLLOW = 0.95
LOW25_ALPHA = 0.95

# ============================================================
# Seasonal helpers
# ============================================================
SHORT_MAP = {
    "DJF": [12, 1, 2],
    "JF":  [1, 2],
    "FM":  [2, 3],
    "MA":  [3, 4],
    "NDJ": [11, 12, 1],
    "JFM": [1, 2, 3],
    "OND": [10, 11, 12],
}

def is_cross_year_season(season_str):
    months = SHORT_MAP.get(season_str, [12, 1, 2])
    return any(m in [11, 12] for m in months) and any(m in [1, 2, 3, 4, 5] for m in months)

def get_seasonal_data(da, season_str, method='mean'):
    """
    处理跨年拼接并返回按 target year 统计的序列
    season_str: "DJF", "MA", ...
    method: 'mean' or 'min'
    """
    months = SHORT_MAP.get(season_str, [12, 1, 2])
    ts = da.to_series()
    all_years = np.unique(da.time.dt.year.values.astype(int))
    results = {}
    cross_year = is_cross_year_season(season_str)

    for yr in all_years:
        try:
            if cross_year:
                early_months = [m for m in months if m < 6]
                late_months  = [m for m in months if m >= 6]

                parts = []
                for m in late_months:
                    parts.append(ts[f"{int(yr-1):04d}-{m:02d}"])
                for m in early_months:
                    parts.append(ts[f"{int(yr):04d}-{m:02d}"])
                combined = pd.concat(parts)
            else:
                combined = pd.concat([ts[f"{int(yr):04d}-{m:02d}"] for m in months])

            # 宽松检查：每月至少 ~28 天
            if len(combined) < len(months) * 28:
                continue

            val = combined.mean() if method == "mean" else combined.min()
            results[int(yr)] = float(val)

        except Exception:
            continue

    if len(results) == 0:
        return xr.DataArray([], coords={"year": []}, dims=("year",))

    return pd.Series(results).to_xarray().rename({"index": "year"})

# ============================================================
# Data helpers
# ============================================================
def select_100hpa_level(da_plev):
    """
    自动兼容 plev 为 Pa 或 hPa
    """
    pvals = np.asarray(da_plev["plev"].values, dtype=float)
    if np.nanmax(np.abs(pvals)) > 2000:
        target = 10000.0   # Pa
    else:
        target = 100.0     # hPa
    return da_plev.sel(plev=target, method="nearest")

def sel_latband(da, lat0=40.0, lat1=80.0, lat_name="lat"):
    lat = da[lat_name]
    descending = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if descending else slice(lat0, lat1)
    return da.sel({lat_name: slc})

def assert_daily_unique(da, name=""):
    da = da.sortby("time")
    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)
    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError(f"{name}: duplicated daily timestamps detected.")
    return da

def load_low25_years(txt_path):
    if txt_path and os.path.exists(txt_path):
        return set(np.loadtxt(txt_path, dtype=int).reshape(-1))
    return set()

def prep_one_exp_raw(exp, ep_nc, o3_nc, low25_txt,
                     fz_season="DJF", fz_method="mean",
                     o3_season="MA", o3_method="min",
                     apply_o3_5d=True, lat_band=(40, 80)):
    """
    返回 raw dataframe:
      exp, year, x_raw, y_raw, is_low25
    """
    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    var_fz = "Fz" if "Fz" in ds_ep.data_vars else "EP2_cart"

    # ---- Fz raw daily ----
    fz_raw = ds_ep[var_fz]
    if "plev" in fz_raw.coords:
        fz_raw = select_100hpa_level(fz_raw)

    if "lat" in fz_raw.coords:
        fz_raw = sel_latband(fz_raw, lat_band[0], lat_band[1], "lat")
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    # 与你前面逻辑一致：使用 -Fz
    fz_raw = -1.0 * fz_raw
    fz_raw = assert_daily_unique(fz_raw, name=f"{exp}_Fz")

    # ---- O3 raw daily ----
    da_o3 = assert_daily_unique(da_o3, name=f"{exp}_O3")
    if apply_o3_5d:
        da_o3 = da_o3.rolling(time=5, center=True, min_periods=5).mean()

    # ---- seasonal aggregation ----
    x_raw = get_seasonal_data(fz_raw, fz_season, method=fz_method)
    y_raw = get_seasonal_data(da_o3,  o3_season, method=o3_method)

    # 拼接型实验在跨年 Fz season 中去掉 bridge year
    if (exp in BRIDGE_EXPERIMENTS) and is_cross_year_season(fz_season):
        if BRIDGE_YEAR in x_raw.year.values:
            x_raw = x_raw.sel(year=(x_raw.year != BRIDGE_YEAR))
            print(f"[{exp}] Removed bridge year {BRIDGE_YEAR} for cross-year Fz season ({fz_season}).")

    common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
    x_raw = x_raw.sel(year=common_years)
    y_raw = y_raw.sel(year=common_years)

    low25_set = load_low25_years(low25_txt)

    df = pd.DataFrame({
        "exp": exp,
        "year": x_raw.year.values.astype(int),
        "x_raw": x_raw.values.astype(float),
        "y_raw": y_raw.values.astype(float),
    })
    df["is_low25"] = df["year"].isin(low25_set)

    ds_ep.close()
    da_o3.close()

    return df

def finalize_raw_df(df):
    """
    不做标准化，只做绘图用缩放
    """
    df = df.copy().dropna(subset=["x_raw", "y_raw"])
    df["x"] = df["x_raw"] * X_PLOT_SCALE
    df["y"] = df["y_raw"]
    return df

def build_all_groups():
    # ---------------- individual experiments ----------------
    df_merra = prep_one_exp_raw(
        "MERRA2",
        FILE_MERRA_EP,
        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
        os.path.join(DATA_BASE, "MERRA2", "low25pct_years_30_70hPa.txt"),
        fz_season=FZ_SEASON, fz_method=FZ_METHOD,
        o3_season=O3_SEASON, o3_method=O3_METHOD,
        apply_o3_5d=APPLY_O3_5D, lat_band=LAT_BAND
    )

    df_bwcn = prep_one_exp_raw(
        "BWCN",
        FILE_BWCN_EP,
        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "BWCN", "low25pct_years_30_70hPa.txt"),
        fz_season=FZ_SEASON, fz_method=FZ_METHOD,
        o3_season=O3_SEASON, o3_method=O3_METHOD,
        apply_o3_5d=APPLY_O3_5D, lat_band=LAT_BAND
    )

    df_b2000 = prep_one_exp_raw(
        "B2000WCN",
        FILE_B2000_EP,
        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN", "low25pct_years_30_70hPa.txt"),
        fz_season=FZ_SEASON, fz_method=FZ_METHOD,
        o3_season=O3_SEASON, o3_method=O3_METHOD,
        apply_o3_5d=APPLY_O3_5D, lat_band=LAT_BAND
    )

    df_clim3d = prep_one_exp_raw(
        "B2000WCN_NOCOUPL",
        FILE_NOCOUPL_EP,
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "low25pct_years_30_70hPa.txt"),
        fz_season=FZ_SEASON, fz_method=FZ_METHOD,
        o3_season=O3_SEASON, o3_method=O3_METHOD,
        apply_o3_5d=APPLY_O3_5D, lat_band=LAT_BAND
    )

    # ---------------- groups ----------------
    df_merra = finalize_raw_df(df_merra)
    df_int3d = finalize_raw_df(pd.concat([df_bwcn, df_b2000], ignore_index=True))
    df_clim3d = finalize_raw_df(df_clim3d)

    return df_merra, df_int3d, df_clim3d

def compute_global_limits(dfs):
    if USE_MANUAL_AXIS_LIMITS:
        return MANUAL_XLIM, MANUAL_YLIM

    all_x = np.concatenate([df["x"].values for df in dfs if df is not None and len(df) > 0])
    all_y = np.concatenate([df["y"].values for df in dfs if df is not None and len(df) > 0])

    xlim = (np.nanmin(all_x) - X_PAD, np.nanmax(all_x) + X_PAD)
    ylim = (np.nanmin(all_y) - Y_PAD, np.nanmax(all_y) + Y_PAD)
    return xlim, ylim

def select_topN_lowest_o3(df, topn=5):
    if df is None or len(df) == 0:
        return df.iloc[[]].copy()
    return df.sort_values("y_raw", ascending=True).head(topn).copy()

# ============================================================
# Plot helpers
# ============================================================
def make_panel_legend(ax, panel_name, top_df):
    handles = []
    labels = []

    if panel_name == "INT-3D":
        handles.extend([
            Line2D([0], [0], marker="o", color="none",
                   markerfacecolor=GRAY_FILL, markeredgecolor="none",
                   alpha=BASE_ALPHA_FILLED, markersize=5.5),
            Line2D([0], [0], marker="o", color="none",
                   markerfacecolor="none", markeredgecolor=GRAY_EDGE,
                   alpha=1.0, markersize=5.5, linewidth=0),
            Line2D([0], [0], marker="o", color="none",
                   markerfacecolor=LOW25_COLOR, markeredgecolor="k",
                   markeredgewidth=0.5, markersize=5.8),
            Line2D([0], [0], marker="o", color="none",
                   markerfacecolor="none", markeredgecolor=LOW25_COLOR,
                   markeredgewidth=1.2, markersize=5.8),
        ])
        labels.extend([
            "B2000WCN (all years)",
            "BWCN (all years)",
            "B2000WCN low25",
            "BWCN low25",
        ])
    else:
        handles.extend([
            Line2D([0], [0], marker="o", color="none",
                   markerfacecolor=GRAY_FILL, markeredgecolor="none",
                   alpha=BASE_ALPHA_FILLED, markersize=5.5),
            Line2D([0], [0], marker="o", color="none",
                   markerfacecolor=LOW25_COLOR, markeredgecolor="k",
                   markeredgewidth=0.5, markersize=5.8),
        ])
        labels.extend([
            "All years",
            "low25 years",
        ])

    for i, (_, r) in enumerate(top_df.iterrows()):
        c = EXTREME_COLORS[i % len(EXTREME_COLORS)]
        if r["exp"] == "BWCN":
            h = Line2D([0], [0], marker="o", color="none",
                       markerfacecolor="none", markeredgecolor=c,
                       markeredgewidth=1.8, markersize=7.0)
            lbl = f"Top{i+1}: BWCN-{int(r['year']):04d}"
        else:
            h = Line2D([0], [0], marker="o", color="none",
                       markerfacecolor=c, markeredgecolor="k",
                       markeredgewidth=0.7, markersize=7.0)
            if panel_name == "INT-3D":
                lbl = f"Top{i+1}: {int(r['year']):04d}"
            else:
                lbl = f"Top{i+1}: {int(r['year']):04d}"
        handles.append(h)
        labels.append(lbl)

    ax.legend(
        handles, labels,
        loc="lower left",
        fontsize=LEGEND_SIZE,
        frameon=False,
        handletextpad=0.35,
        borderpad=0.2,
        labelspacing=0.28
    )

def draw_one_panel(ax, df, panel_name, xlim, ylim, show_ylabel=False):
    ax.grid(True, linestyle=":", alpha=0.55)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # 可选：如果你想保留零线就保留
    ax.axhline(0, color="k", lw=0.8, alpha=0.35)
    ax.axvline(0, color="k", lw=0.8, alpha=0.35)

    if df is None or len(df) == 0:
        ax.set_title(panel_name, fontsize=TITLE_SIZE, fontweight="bold")
        ax.text(0.5, 0.5, "No valid data", transform=ax.transAxes,
                ha="center", va="center", fontsize=11)
        return

    df = df.copy()

    # masks
    is_bwcn = df["exp"].eq("BWCN")
    is_low25 = df["is_low25"].values
    is_base = ~is_low25

    # ---------------- base points ----------------
    # filled gray: all years except BWCN
    m = is_base & (~is_bwcn)
    if np.any(m):
        ax.scatter(
            df.loc[m, "x"], df.loc[m, "y"],
            s=BASE_SIZE, alpha=BASE_ALPHA_FILLED, color=GRAY_FILL,
            edgecolors="none", zorder=2
        )

    # hollow gray: BWCN
    m = is_base & is_bwcn
    if np.any(m):
        ax.scatter(
            df.loc[m, "x"], df.loc[m, "y"],
            s=BASE_SIZE, alpha=BASE_ALPHA_HOLLOW,
            facecolors="none", edgecolors=GRAY_EDGE, linewidths=0.8, zorder=2
        )

    # ---------------- low25 ----------------
    # filled red
    m = is_low25 & (~is_bwcn)
    if np.any(m):
        ax.scatter(
            df.loc[m, "x"], df.loc[m, "y"],
            s=LOW25_SIZE, alpha=LOW25_ALPHA,
            color=LOW25_COLOR, edgecolors="k", linewidths=0.45, zorder=5
        )

    # hollow red for BWCN
    m = is_low25 & is_bwcn
    if np.any(m):
        ax.scatter(
            df.loc[m, "x"], df.loc[m, "y"],
            s=LOW25_SIZE, alpha=1.0,
            facecolors="none", edgecolors=LOW25_COLOR, linewidths=1.2, zorder=5
        )

    # ---------------- top5 lowest O3 ----------------
    top_df = select_topN_lowest_o3(df, topn=EXTREME_TOPN) if MARK_EXTREMES else df.iloc[[]].copy()

    for i, (_, r) in enumerate(top_df.iterrows()):
        c = EXTREME_COLORS[i % len(EXTREME_COLORS)]
        if r["exp"] == "BWCN":
            ax.scatter(
                r["x"], r["y"],
                s=EXTREME_SIZE, facecolors="none", edgecolors=c,
                linewidths=2.0, zorder=10
            )
        else:
            ax.scatter(
                r["x"], r["y"],
                s=EXTREME_SIZE, color=c, edgecolors="k",
                linewidths=0.8, zorder=10
            )

    # ---------------- stats ----------------
    # ALL
    n_all = len(df)
    if n_all >= 3:
        try:
            r_all, p_all = pearsonr(df["x"].values, df["y"].values)
            txt_all = f"ALL\nr = {r_all:.3f}\np = {p_all:.2e}\nN = {n_all}"
        except Exception:
            txt_all = f"ALL\nN = {n_all}"
    else:
        txt_all = f"ALL\nN = {n_all}"

    ax.text(
        0.03, 0.97, txt_all,
        transform=ax.transAxes, va="top", ha="left",
        fontsize=TEXT_SIZE,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.72, edgecolor="none")
    )

    # LOW25
    low_df = df[df["is_low25"]].copy()
    n_low = len(low_df)
    if n_low >= 3:
        try:
            r_low, p_low = pearsonr(low_df["x"].values, low_df["y"].values)
            txt_low = f"LOW25\nr = {r_low:.3f}\np = {p_low:.2e}\nN = {n_low}"
        except Exception:
            txt_low = f"LOW25\nN = {n_low}"
    else:
        txt_low = f"LOW25\nN = {n_low}"

    ax.text(
        0.97, 0.97, txt_low,
        transform=ax.transAxes, va="top", ha="right",
        fontsize=TEXT_SIZE, color=LOW25_COLOR,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.72, edgecolor="none")
    )

    # title
    ax.set_title(panel_name, fontsize=TITLE_SIZE, fontweight="bold")

    # ticks
    ax.tick_params(axis="both", labelsize=TICK_SIZE)

    # ylabel only on left panel
    if not show_ylabel:
        ax.set_ylabel("")
        ax.tick_params(axis="y", left=False, labelleft=False)
        ax.spines["left"].set_visible(False)

    # legend
    make_panel_legend(ax, panel_name, top_df)

# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    print("Building data tables...")

    df_merra, df_int3d, df_clim3d = build_all_groups()

    xlim, ylim = compute_global_limits([df_merra, df_int3d, df_clim3d])

    print("xlim =", xlim)
    print("ylim =", ylim)

    fig, axes = plt.subplots(
        1, 3, figsize=FIGSIZE, sharey=True
    )
    plt.subplots_adjust(wspace=0.08)

    draw_one_panel(axes[0], df_merra,  "MERRA2",  xlim, ylim, show_ylabel=True)
    draw_one_panel(axes[1], df_int3d,  "INT-3D",  xlim, ylim, show_ylabel=False)
    draw_one_panel(axes[2], df_clim3d, "Clim-3D", xlim, ylim, show_ylabel=False)

    # shared labels
    fig.supxlabel(
        r"DJF mean $-F_z$ at 100 hPa (40–80°N mean) ($10^{-3}$ hPa m s$^{-2}$)",
        fontsize=LABEL_SIZE, y=0.015
    )
    fig.supylabel(
        r"Mar–Apr minimum O$_3$ (DU)",
        fontsize=LABEL_SIZE, x=0.04
    )

    # 如果你觉得总标题多余，可以直接删掉这行
    # fig.suptitle("DJF EP flux vs Mar–Apr minimum ozone", fontsize=15, fontweight="bold", y=1.02)

    base = "Scatter_DJFmean_Fz_vs_MAminO3_RAW_MERRA2_INT3D_Clim3D"

    png_path = os.path.join(OUT_DIR, base + ".png")
    tif_path = os.path.join(OUT_DIR, base + ".tif")
    pdf_path = os.path.join(OUT_DIR, base + ".pdf")

    fig.savefig(png_path, dpi=SAVE_DPI, bbox_inches="tight")
    fig.savefig(tif_path, dpi=SAVE_DPI, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")

    if SHOW_FIG:
        plt.show()
    else:
        plt.close(fig)

    print("Saved:")
    print("  PNG:", png_path)
    print("  TIF:", tif_path)
    print("  PDF:", pdf_path)